# Rugby Rulebook Hierarchical Tree Extraction

For each rulebook year-range, this notebook parses the cleaned rulebook text
into a **nested YAML tree** representing its heading structure (parts, laws,
sub-laws, letter/number sub-points, penalties, notes, etc.), saves that YAML,
then loads it back, converts it into a `networkx` graph, and draws it as a
radial ("twopi") tree layout.

Each year-range gets two versions:
- **Hierarchical (nested)**: every heading level in the text becomes its own
  tree depth.
- **Flattened**: some heading levels are collapsed together, producing a
  shallower tree — useful for comparing structural complexity without every
  fine-grained sub-level.

The parsing logic (which regex patterns map to which tree depth) is
**different for every year-range**, because the rulebooks' formatting
conventions changed over time (e.g. `LAW 12` vs `Law 12`, `(a)` vs `a.`,
lettered sub-points vs none). Each section below starts with the original
author's notes on the pattern for that era, followed by the parser, then the
code that runs it on that range's text files and visualizes the result.

## Requirements

```bash
pip install networkx pyyaml matplotlib numpy pygraphviz
```
`pygraphviz` requires Graphviz to be installed on your system (e.g.
`brew install graphviz` on macOS, `apt install graphviz libgraphviz-dev` on
Debian/Ubuntu) *before* `pip install pygraphviz`.

## Data

This notebook expects a `data/` folder (next to the notebook) laid out as:

```
data/
├── rugby_test_matches_per_year_backfilled.csv
├── rugby_laws_per_year.csv
├── hierarchical_trees/
│   ├── final_scrum_branches/
│   ├── flattened/                  (generated)
│   └── original/                   (generated)
├── interdependencies/
│   ├── laws/                  (generated)
│   └── laws_plan_defs/        (generated)  
└── text_files/
    └── cleaned_relavent_texts/
        ├── 1845/
        ├── 1874-1889/
        ├── 1894-1924/
        ├── 1929/
        ├── 1934-1949/
        ├── 1954-1959/
        ├── 1969-1974/
        ├── 1985-1995/
        ├── 1997-1998/
        ├── 2000-2002/
        ├── 2003-2016/
        └── 2018-2026/
```


## Imports

In [ ]:
import networkx as nx
from pathlib import Path 
import yaml
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout
import re
import string
import numpy as np
import copy
import matplotlib.cm as cm

## Configuration

Update `DATA_DIR` if your data lives somewhere other than next to this
notebook.

In [ ]:
DATA_DIR = Path('data')

TEXTS_DIR = DATA_DIR / 'text_files' / 'cleaned_relavent_texts'
TREES_BASE = DATA_DIR / 'hierarchical_trees'
TREES_DIR = TREES_BASE / 'trees' / 'tree_structures_paragraphs'

## Core helper functions

`create_heirarchical_tree` walks a parsed YAML tree (regardless of which
year-range parser produced it) up to 5 levels deep and turns it into a
`networkx.Graph`, with node color/size set by depth. `generate_graph` then
lays that graph out radially (`graphviz_layout`, `prog='twopi'`) and draws
it, auto-scaling the axes so the tree isn't stretched. Both functions are
reused for every year-range below.

In [ ]:
def create_heirarchical_tree(data):
    colors = cm.viridis(np.linspace(0, 0.85, 10))
    
    G = nx.Graph()
    G.add_node('center', color=colors[0], node_size=300)
    level1_no = 0

    # adds level1 nodes, connecting to center
    for edge in data['(full ruleset)']:
        temp_data1 = data['(full ruleset)']

        level1_node = next(iter(edge))
        level1_node_name = str(level1_node + ' ')
        level1_no += 1
        level2_no = 0

        G.add_node(level1_node_name, color=colors[1], node_size=200)
        G.add_edge('center', level1_node_name)

        #print("LAW:", law_node_name)


        # adds level2 nodes, connecting to level1
        for edge in temp_data1[level1_no-1][level1_node]:
            temp_data2 = temp_data1[level1_no-1][level1_node]

            level2_node = next(iter(edge))
            level2_node_name = str(level1_node_name + level2_node + ' ')
            level2_no += 1
            level3_no = 0

            G.add_node(level2_node_name, color=colors[2], node_size=160)
            G.add_edge(level1_node_name, level2_node_name)

            #print("HEADING:", heading_node_name)


            # adds level3, connecting to level2
            for edge in temp_data2[level2_no-1][level2_node]:
                temp_data3 = temp_data2[level2_no-1][level2_node]

                level3_node = next(iter(edge))
                level3_node_name = str(level2_node_name + level3_node + ' ')
                level3_no += 1
                level4_no = 0

                G.add_node(level3_node_name, color=colors[3], node_size=120)
                G.add_edge(level2_node_name, level3_node_name)

                #print("SUBLAW:", sublaw_node_name)


                # adds level4, connecting to level3
                for edge in temp_data3[level3_no-1][level3_node]:

                    level4_node = next(iter(edge))
                    level4_node_name = str(level3_node_name + level4_node+ ' ')
                    level4_no += 1
                    level5_no = 0

                    G.add_node(level4_node_name, color=colors[4], node_size=80)
                    G.add_edge(level3_node_name, level4_node_name)

                    #print("LOWER:", subsublaw_node_name)

                    # adds level5, connecting to level4
                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node]:
                        level5_node = next(iter(edge))
                        level5_node_name = str(level4_node_name + level5_node + ' ')
                        level5_no += 1
                        level6_no = 0

                        G.add_node(level5_node_name, color=colors[5], node_size=60)
                        G.add_edge(level4_node_name, level5_node_name)

                        #print("ROMAN:", subsubsublaw_node_name)

                        # adds level6, connecting to level5
                        for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node]:
                            level6_node = next(iter(edge))
                            level6_node_name = str(level5_node_name + level6_node + ' ')
                            level6_no += 1
                            level7_no = 0

                            G.add_node(level6_node_name, color=colors[6], node_size=40)
                            G.add_edge(level5_node_name, level6_node_name)


                            # adds level7, connecting to level6
                            for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node]:
                                level7_node = next(iter(edge))
                                level7_node_name = str(level6_node_name + level7_node + ' ')
                                level7_no += 1
                                level8_no = 0

                                G.add_node(level7_node_name, color=colors[7], node_size=35)
                                G.add_edge(level6_node_name, level7_node_name)

                                # adds level8, connecting to level7
                                for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node]:
                                    level8_node = next(iter(edge))
                                    level8_node_name = str(level7_node_name + level8_node + ' ')
                                    level8_no += 1
                                    level9_no = 0

                                    G.add_node(level8_node_name, color=colors[8], node_size=30)
                                    G.add_edge(level7_node_name, level8_node_name)

                                    # adds level9, connecting to level8
                                    for edge in temp_data3[level3_no-1][level3_node][level4_no-1][level4_node][level5_no-1][level5_node][level6_no-1][level6_node][level7_no-1][level7_node][level8_no-1][level8_node]:
                                        level9_node = str(edge)
                                        level9_node_name = str(level8_node_name + level9_node + ' ')
                                        level9_no += 1
                                        level10_no = 0

                                        G.add_node(level9_node_name, color=colors[9], node_size=25)
                                        G.add_edge(level8_node_name, level9_node_name)



    return G
        


In [ ]:
def generate_graph(G, year):
    pos = graphviz_layout(G, prog='twopi', root='center', args='-Groot="center" -Gnodesep=0.4 -Granksep=1')
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_sizes = [G.nodes[node]['node_size'] for node in G.nodes()]


    fig, ax = plt.subplots(figsize=(16,16))
    #ax.set_aspect('scaled')
    nx.draw(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, width=1, edge_color='dimgrey')

    # make size of axes so doesn't pull diagram
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    max_range = max(max(xs) - min(xs), max(ys) - min(ys))
    center_x = (min(xs) + max(xs)) / 2
    center_y = (min(ys) + max(ys)) / 2
    padding = 0.05 * max_range
    half = (max_range / 2) + padding
    ax.set_xlim(center_x - half, center_x + half)
    ax.set_ylim(center_y - half, center_y + half)
    ax.set_aspect("equal")

    plt.title(f'{year}', size=40)
    #plt.annotate(f'{year}',xy=(0, 1), xytext=(10, -10), xycoords='axes fraction', textcoords='offset points', ha='left', va='top', size=40)

Leftover working notes on the general depth-calculation approach
(kept as originally written; superseded by the year-specific notes in each
section below).

## 2018-2026

In [ ]:
# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. definitions + laws
 2. all caps subheading
 3. paragraph (sometimes)
 4. #.
 5. lowercase.
 6. lowercase roman numeral. 
 7. sanction
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# get LAW 19
def create_yamls_2018_2026(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    level8_no = 0
    found_headings = False
    in_definitions = False
    in_laws = False
    penalty3 = False
    penalty4 = False
    penalty5 = False
    paragraph2 = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False
    paragraph8 = False
    paragraph_no = 0

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})



    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_laws:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            continue


        # adds defintions
        if (re.search(r'^(?!(\s)).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
       


        # begins laws
        # always level 1
        if re.search(r'^(LAW 1)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_laws = True
            in_definitions = False
            paragraph_no = 0

        # returns LAWS
        if re.search(r'(LAW)\s[0-9]', temp_text) and not in_definitions:
            law_no += 1
            law_string = str('LAW ' + str(law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({law_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph_no = 0
            sublaw_no_label = 0
            found_headings=False

        # subheadingless
        # return arabic numbers prior to any subheadings --> level 2
        if re.search(r'([0-9]|[0-9][0-9])\.\s', temp_text) and law_no>=1 and not found_headings and not in_definitions:
            in_alphabet = False
            penalty3 = False
            penalty4 = False
            level2_no += 1
            sublaw_no_label += 1
            level2_string = str(sublaw_no_label)
            yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph_no = 0
            paragraph2 = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False

        # adds lowercase letters (subsublaws) --> level 3
        subsublaw_letters = list(string.ascii_letters)
        if re.search(r'^[a-z]\.\s', temp_text) and law_no>=1 and level2_no>=1 and not found_headings and not in_definitions:
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'^i\.\s', temp_text):
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif re.match(r'^i\.\s', temp_text) and {'h': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string]:
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0 
                level5_no = 0 
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # returns sanctions 
        if (re.search(r'^(Sanction:)\s', temp_text) or re.search(r'\s(Sanction:)\s', temp_text)) and level2_no>=1 and not found_headings and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 3
            if not in_alphabet:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 4
            elif in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False


        # subheadings now
        # returns uppercase SUBHEADINGS --> level 2
        if re.search(r'^(?!LAW)[A-Z][A-Z]', temp_text) and law_no >=1 and not in_definitions:
            paragraph_no = 0
            found_headings = True
            level2_no += 1
            level2_string = str('HEADING' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph2 = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False

        # adds arabic numbers (sublaws) --> level 3
        if re.search(r'([0-9]|[0-9][0-9])\.\s', temp_text) and law_no>=1 and level2_no>=1 and found_headings and not in_definitions:
            paragraph_no = 0
            in_alphabet = False
            penalty4 = False
            penalty5 = False
            level3_no += 1
            sublaw_no_label += 1
            level3_string = str(sublaw_no_label)
            yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0 
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False

        # adds lowercase letters (subsublaws) --> level 4
        if re.search(r'^[a-z]\.\s', temp_text) and law_no>=1 and level3_no>=1 and found_headings and not in_definitions:
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'^i\.\s', temp_text):
                in_alphabet = True
                level4_no += 1
                level4_string = subsublaw_letters[level4_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif re.match(r'^i\.\s', temp_text) and {'h': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                in_alphabet = True
                level4_letters = list(string.ascii_letters)
                level4_no += 1
                level4_string = subsublaw_letters[level4_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0 
                level6_no = 0               
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # returns sanctions 
        if (re.search(r'^(Sanction:)\s', temp_text) or re.search(r'\s(Sanction:)\s', temp_text)) and level2_no>=1 and level3_no>=1 and found_headings and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # adds lowercase roman numerals (subsubsublaws)
        # STILL NEED TO FIX -- not detecting v. 
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and found_headings and not in_definitions:
            paragraph_no = 0
            if (not penalty3 or (penalty3 and not re.search(r'^ii\.\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^ii\.\s', temp_text)))and (not penalty5 or (penalty5 and not re.search(r'^ii\.\s', temp_text))):
                penalty3 = False
                penalty4 = False
                penalty5 = False
                level5_no += 2
                level5_string = str('ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['i. '].append({'P1':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                level7_no = 0
            elif penalty3:
                level4_no += 2
                level4_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-2]['penalty i. '].append({'P1':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})  
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty4:
                level5_no += 2
                level5_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['penalty i. '].append({'P1':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})            
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty5:
                level6_no += 2
                level6_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['penalty i. '].append({'P1':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})    
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
        if (re.search(r'(iii|iv|v)\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and found_headings and not in_definitions:
            paragraph_no = 0
            if not penalty3 and not penalty4 and not penalty5:
                level5_no += 1
                level5_string = subsubsublaw_labels[level5_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(subsubsublaw_labels[level3_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(subsubsublaw_labels[level4_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty ' + str(subsubsublaw_labels[level5_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False


        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|[A-Z][A-Z]|i\.\s)).*', temp_text)) and law_no>=1 and level2_no>=1 and not in_definitions:
            if (level2_no==0 or paragraph2) and level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph2 = True
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string].append({level2_string: []})
            elif (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7):
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
 
  

    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2018_2026 = (TEXTS_DIR / '2018-2026')
final_path_2018_2026 = (TREES_DIR / '2018-2026' / 'clean_')

for file_path in initial_path_2018_2026.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_2018_2026(file, final_path_2018_2026, year)
        

In [ ]:
yaml_files_path_2018_2026 = (TREES_DIR / '2018-2026')

for file_path in yaml_files_path_2018_2026.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
# attempting to add #-less headings
def create_yamls_2018_2026_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    law_no = 0
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    in_alphabet = False
    in_definitions = True
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})

    level0_no = 2
    level0_string = str('DEFINITIONS')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # adds defintions
        if (re.search(r'^(?!(DEFINITIONS|\s)).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       
        

        # main ruleset section
        if re.search(r'^(LAW|Law)\s1(\.|\s|:)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            law_no = 0
            level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW|Law)\s[0-9]', temp_text)) and not in_definitions:
            law_no += 1
            real_law_no += 1
            law_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({law_string: []})
            in_laws = True
            in_definitions = False
            level2_no = 0
    
        # return arabic numbers prior to any subheadings --> level 2
        if re.search(r'([0-9]|[0-9][0-9])\.\s', temp_text) and law_no>=1 and not in_definitions:
            in_alphabet = False
            penalty3 = False
            penalty4 = False
            level2_no += 1
            level2_string = str('sublaw ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            paragraph_no = 0

        # adds lowercase letters (subsublaws) --> level 3
        subsublaw_letters = list(string.ascii_letters)
        if re.search(r'^[a-z]\.\s', temp_text) and law_no>=1 and level2_no>=1 and not in_definitions:
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'^i\.\s', temp_text):
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
            elif re.match(r'^i\.\s', temp_text) and {'h': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string]:
                in_alphabet = True
                level3_no += 1
                level3_string = subsublaw_letters[level3_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0  
                level5_no = 0

        # returns sanctions 
        if (re.search(r'^(Sanction:)\s', temp_text) or re.search(r'\s(Sanction:)\s', temp_text)) and level2_no>=1 and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 3
            if not in_alphabet:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0


        # adds lowercase roman numerals (subsubsublaws)
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            if (not penalty3 or (penalty3 and not re.search(r'^ii\.\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^ii\.\s', temp_text))):
                penalty3 = False
                penalty4 = False
                level4_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'ii. ': []})
                level5_no = 0
            elif penalty3:
                level3_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({'penalty ii. ': []})
                level4_no = 0
                level5_no = 0
            elif penalty4:
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ': []})
                level5_no = 0
        if (re.search(r'(iii|iv|v)\.\s', temp_text)) and law_no >=1 and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            if not penalty3 and not penalty4:
                level4_no += 1
                level4_string = subsubsublaw_labels[level4_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(subsubsublaw_labels[level3_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(subsubsublaw_labels[level4_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|PRINCIPLE|Sanction|Excep|o|•|\*|-|[0-9]|\(|\s|i|v|x|DEF|[a-z]\.\s|[A-Z][A-Z])).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_definitions:
            if law_no==0 and level3_no==0 and level4_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    law_no += 1
                    law_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({law_string: []})
            elif law_no!=0 and level3_no==0 and level4_no==0:
                paragraph_no += 1
                if paragraph_no>=1:
                    level2_no += 1
                    level2_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string].append({level2_string: []})
            elif law_no!=0 and level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string].append({level3_string: []})
            elif law_no!=0 and level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][law_no-1][law_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            


    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2018_2026 = (TEXTS_DIR / '2018-2026')
final_path_2018_2026_flat = (TREES_DIR / '2018-2026flat' / 'clean_')

for file_path in initial_path_2018_2026.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2018-2026', '')
        
        create_yamls_2018_2026_flattened(file, final_path_2018_2026_flat, year)

In [ ]:
yaml_files_path_2018_2026flat = (TREES_DIR / '2018-2026flat')

for file_path in yaml_files_path_2018_2026flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 2003-2016

The author's notes on this era's heading/depth pattern:

In [ ]:


# 2003-2018

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. definitions, before vs. during the match (section)
 2. method of playing the match, in the field of play, restarts, in-goal (for during) (subsection)
 3. laws  as Law # but some LAW #
 4. capital sections  #.A, #.B (not always)
 5. definitions before sublaws - DEFINITIONS
 6. sublaws #.# or #.A.# - ALL CAPS HEADING
 7. (a), (b), (c)
 8. bullets (sometimes may be lowercase-less bullets)
 9. lowercase roman numeral. 
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# adding exceptions rn
def create_yamls_2003_2016(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    level8_no = 0
    paragraph_no = 0
    in_definitions = False
    in_laws = False
    paragraph2 = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False
    paragraph8 = False
    paragraph_no = 0

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})



    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_laws:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            continue


        # adds defintions
        if (re.search(r'^(?!(\s|BEFORE|(Rugby Union))).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
       


        # before the match -- no subsections
        # returns section titles
        # always level 1
        if re.search(r'^(BEFORE THE MATCH)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_laws = True
            in_definitions = False
            level1_no += 1
            level1_string = str('SECTION ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            subsections = False
            level2_no = 0
            level3_no = 0
            paragraph_no = 0

        # returns laws --> level 2
        if (re.search(r'^(Law)\s[0-9]', temp_text) or re.search(r'^(LAW)\s[0-9]', temp_text)) and level1_no>=1 and not subsections and not in_definitions:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            letter_sections = False
            letter_no = 0
            paragraph_no = 0
            paragraph2 = False
            paragraph3 = False
            paragraph5 = False
            paragraph4 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns letter sections --> level 3 if present
        if re.search(fr'^{real_law_no}\.(A|B|C|D)\s', temp_text) and level1_no>=1 and level2_no>=1 and not subsections and not in_definitions:
            letter_sections = True
            level3_no += 1
            letter_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph8 = False
            paragraph7 = False

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^{real_law_no}\.(A|B|C|D)\.(1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16)+\s', temp_text) or re.search(fr'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and letter_sections and not in_definitions:
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections present --> level 5
        if re.search(r'\([a-t]\)', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            in_alphabet = True
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level5_no += 1
                level5_string = str('subsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]:
                level5_no += 1
                level5_string = str('subsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            # bullets BEFORE lowercase letters --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 6
            else:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns roman numerals -- letter sections present -- level 6
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and not subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            level6_no += 2
            level6_string = subsubsublaw_labels[level6_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['i. '].append({'P1':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
            level7_no = 0
            level8_no = 0
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False
        elif (re.search(r'(iii|iv|v)\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            level6_no += 1
            level6_string = subsubsublaw_labels[level6_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]}) 
            level7_no = 0
            level8_no = 0
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns sanctions -- letter sections present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^{real_law_no}\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            in_alphabet = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections NOT present --> level 4
        if re.search(r'\([a-t]\)', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            in_alphabet = True
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            # bullets BEFORE lowercase letters --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 5
            else:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns roman numerals -- letter sections NOT present --> level 5
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            level5_no += 2
            level5_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['i. '].append({'P1':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False
        elif (re.search(r'(iii|iv|v)\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            level5_no += 1
            level5_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]}) 
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('penalty/exception ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False


        # after the match -- subsections
        # returns section titles
        # always level 1
        if re.search(r'^(DURING THE MATCH)', temp_text) and not in_definitions:
            paragraph_no = 0
            level1_no += 1
            level1_string = str('SECTION ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            subsections = True
            level2_no = 0

        # returns subsections --> level 2
        if re.search(r'^(Method of Playing the Match|In the Field of play|Restarts|In-Goal)$', temp_text) and level1_no>=1 and subsections and not in_definitions:
            paragraph_no = 0
            level2_no += 1
            level2_string = str('section ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0

        # returns laws --> level 3
        if (re.search(r'^(Law)\s[0-9]', temp_text) or re.search(r'^(LAW)\s[0-9]', temp_text)) and level1_no>=1 and level2_no>=1 and subsections and not in_definitions:
            paragraph_no = 0
            real_law_no += 1
            level3_no += 1
            level3_string = str('LAW ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            letter_sections = False
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            letter_no = 0
            level8_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns letter sections --> level 4 if present
        if re.search(fr'^{real_law_no}\.(A|B|C|D|E)\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and subsections and not in_definitions:
            paragraph_no = 0
            letter_sections = True
            letter_no += 1
            level4_no += 1
            level4_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns defintions or sublaws -- letter sections present --> level 5
        if (re.search(fr'^{real_law_no}\.(A|B|C|D|E)\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^(DEFINITION|Definition)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            letter_sections = True
            in_alphabet = False
            level5_no += 1
            level5_string = str('sublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections present --> level 6
        if re.search(r'\([a-t]\)', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and letter_sections and not in_definitions:
            in_alphabet = True
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level6_no += 1
                level6_string = str('subsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string]:
                level6_no += 1
                level6_string = str('subsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            # bullets BEFORE lowercase letters --> level 6
            if not in_alphabet:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 7
            else:
                level7_no += 1
                level7_string = str('subsubsublaw ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False
                paragraph8 = False

        # returns roman numerals -- letter sections present -- level 7
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and level6_no>=1 and subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            level7_no += 2
            level7_string = subsubsublaw_labels[level7_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-2]['i. '].append({'P1':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
            level8_no = 0
            paragraph7 = False
            paragraph8 = False
        elif (re.search(r'(iii|iv|v)\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            level7_no += 1
            level7_string = subsubsublaw_labels[level7_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]}) 
            level8_no = 0
            paragraph7 = False
            paragraph8 = False

        # returns sanctions -- letter sections present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and subsections and letter_sections and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 6
            if not in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 7
            elif in_alphabet:
                level7_no += 1
                level7_string = str('penalty/exception ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False
                paragraph8 = False


        # returns defintions or sublaws -- letter sections NOT present --> level 4
        if (re.search(fr'^{real_law_no}\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^(DEFINITION|Definition)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections NOT present --> level 5
        if re.search(r'\([a-r]\)', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and not letter_sections and not in_definitions:
            in_alphabet = True
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level5_no += 1
                level5_string = str('subsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]:
                level5_no += 1
                level5_string = str('subsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            # bullets BEFORE lowercase letters --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 6
            else:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns roman numerals -- letter sections present --> level 6
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^ii\.\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            level6_no += 2
            level6_string = subsubsublaw_labels[level6_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['i. '].append({'P1':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
            level7_no = 0
            level8_no = 0
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False
        elif (re.search(r'(iii|iv|v)\.\s', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            level6_no += 1
            level6_string = subsubsublaw_labels[level6_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]}) 
            level7_no = 0
            level8_no = 0
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and subsections and not letter_sections and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

    

        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|INSTRUCTIONS)).*', temp_text)) and len(temp_text)>=20 and level1_no>=1 and level2_no>=1 and real_law_no>=1 and not in_definitions:
            if (level2_no==0 or paragraph2) and level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph2 = True
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            elif (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0 and level8_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7) and level8_no==0:
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0 and (level8_no==0 or paragraph8):
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0 and level8_no!=0:
                paragraph9 = True
                paragraph_no += 1
                level9_no += 1
                level9_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string][level8_no-1][level8_string].append({level9_string: []})
  
  
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2003_2016 = (TEXTS_DIR / '2003-2016')
final_path_2003_2016 = (TREES_DIR / '2003-2016' / 'clean_')

for file_path in initial_path_2003_2016.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2003-2016', '')
        
        create_yamls_2003_2016(file, final_path_2003_2016, year)

In [ ]:
yaml_files_path_2003_2016 = (TREES_DIR / '2003-2016')

for file_path in yaml_files_path_2003_2016.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_2003_2016_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    in_definitions = True
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})

    level0_no = 2
    level0_string = str('DEFINITIONS')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # adds defintions
        if (re.search(r'^(?!(LAWS|Definitions|\s|BEFORE)).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       
        

        # main ruleset section
        if re.search(r'^(LAW|Law)\s1(\.|\s)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW|Law)\s[0-9]', temp_text)):
            in_laws = True
            in_definitions = False
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            paragraph_no = 0

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^{real_law_no}\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^{real_law_no}\.(A|B|C|D|E)\.([0-9]|[0-9][0-9])\s', temp_text) or re.search(fr'^(DEFINITION|Definition)', temp_text)) and level0_no==1 and level2_no>=1 and not in_definitions:
            in_alphabet = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            paragraph_no = 0

        # returns lowercase letters -- letter sections NOT present --> level 4
        if re.search(r'\([a-r]\)', temp_text) and level2_no>=1 and level3_no>=1 and not in_definitions:
            in_alphabet = True
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0

            elif re.match(r'\(i\)\s', temp_text) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                level4_no += 1
                level4_string = str('subsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
                level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(•|\*|o)\s', temp_text) and level2_no>=1 and level3_no>=1 and not in_definitions:
            # bullets BEFORE lowercase letters --> level 4
            if level4_no==0:
                paragraph_no = 0
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # bullets AFTER lowercase letters --> level 5
            elif level4_no>=1:
                paragraph_no = 0
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns roman numerals -- letter sections NOT present --> level 5
        subsubsublaw_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^(ii\.|\(ii\))\s', temp_text) or re.search(r'\sii\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not in_definitions:
            paragraph_no = 0
            level5_no += 2
            level5_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            level6_no = 0
        if (re.search(r'(iii|iv|v|\(iii|\(iv|\(v)\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1:
            paragraph_no = 0
            level5_no_no += 1
            subsubsublaw_string = subsubsublaw_labels[level5_no-1]
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({subsubsublaw_string:[]}) 
            level6_no = 0

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('penalty/exception ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|Sanction|Excep|Definit|o|•|\*|-|[0-9]|\(|[a-z]\)|\s|i|v|x|DEF)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_definitions:
            #print(temp_text)
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                if paragraph_no >=2:
                    level2_no += 1
                    level2_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                if paragraph_no >=2:
                    level3_no += 1
                    level3_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})


    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2003_2016 = (TEXTS_DIR / '2003-2016')
final_path_2003_2016_flat = (TREES_DIR / '2003-2016flat' / 'clean_')

for file_path in initial_path_2003_2016.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2003-2016', '')
        
        create_yamls_2003_2016_flattened(file, final_path_2003_2016_flat, year)

In [ ]:
yaml_files_path_2003_2016flat = (TREES_DIR / '2003-2016flat')

for file_path in yaml_files_path_2003_2016flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('2003-2016', '')
        year = year.replace('clean_', '')
        year = year.replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 2000-2002

The author's notes on this era's heading/depth pattern:

In [ ]:


# 2000-2002

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. definitions, before vs. during the match (section)
 2. method of playing the match, in the field of play, restarts, in-goal (for during) (subsection)
 3. laws  as Law # but some LAW #
 4. definitions before sublaws - DEFINITIONS
 5. capital sections  A. CAPS, B. CAPS (not always)
 6. sublaws # or # - ALL CAPS HEADING
 7. ALL CAPS SUBHEADINGS (not often -- or at all?)
 8. (a), (b), (c), including DEFINITIONS
 9. - bullets
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# adding sanctions + penalties
def create_yamls_2000_2002(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    level8_no = 0
    in_definitions = False
    in_laws = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False
    paragraph8 = False
    paragraph_no = 0

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})



    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_laws:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            continue


        # adds defintions
        if (re.search(r'^(?!(\s|BEFORE|(Rugby Union))).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
       
        


        # before the match -- no subsections
        # returns section titles
        # always level 1
        if re.search(r'^(BEFORE THE MATCH)', temp_text):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_laws = True
            in_definitions = False
            level1_no += 1
            level1_string = str('SECTION ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            subsections = False
            level2_no = 0
            level3_no = 0


        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and level1_no>=1 and not subsections and not in_definitions:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            letter_sections = False
            paragraph2 = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False


        # returns letter sections --> level 3 if present
        if re.search(fr'^(A|B|C|D)\.\s[A-Z]', temp_text) and level1_no>=1 and level2_no>=1 and not subsections and not in_definitions:
            letter_sections = True
            level3_no += 1
            level3_string = str('LETTER ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^([0-9]|[0-9][0-9])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level4_no==0)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and letter_sections and not in_definitions:
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letter or definitions -- letter sections present --> level 5
        if (re.search(r'\([a-t]\)', temp_text) or re.search(fr'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            in_alphabet = True
            level5_no += 1
            level5_string = str('subsublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'-\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            # bullets BEFORE lowercase letters --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 6
            else:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False


        # returns sanctions -- letter sections present
        if (re.search(r'^(Penalty:)\s', temp_text) or re.search(r'\s(Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not subsections and letter_sections and not in_definitions:
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
        
        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^([0-9]|[0-9][0-9]])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level3_no==0)) and level1_no>=1 and level2_no>=1 and not subsections and not letter_sections and not in_definitions:
            in_alphabet = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections NOT present --> level 4
        if (re.search(r'\([a-t]\)', temp_text) or re.search(fr'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            in_alphabet = True
            level4_no += 1
            level4_string = str('subsublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns bullets -- letter sections NOT present 
        if re.search(r'-\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            # bullets BEFORE lowercase letters --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 5
            else:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Penalty:)\s', temp_text) or re.search(r'\s(Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and not subsections and not letter_sections and not in_definitions:
            # same level as lowercase --> level 4
            if not in_alphabet:
                level4_no += 1
                level4_string = str('penalty/exception ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False


        # after the match -- subsections
        # returns section titles
        # always level 1
        if re.search(r'^(DURING THE MATCH)', temp_text) and not in_definitions:
            level1_no += 1
            level1_string = str('SECTION ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            subsections = True
            level2_no = 0
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0

        # returns subsections --> level 2
        if re.search(r'^(HOW THE GAME IS PLAYED|IN THE FIELD-OF-PLAY|RESTARTS|IN-GOAL)$', temp_text) and level1_no>=1 and subsections and not in_definitions:
            level2_no += 1
            level2_string = str('section ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph2 = False

        # returns laws --> level 3
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and level1_no>=1 and level2_no>=1 and subsections and not in_definitions:
            real_law_no += 1
            level3_no += 1
            level3_string = str('LAW ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            letter_sections = False
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns letter sections --> level 4 if present
        if re.search(fr'^(A|B|C|D)\.\s[A-Z]', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and subsections and not in_definitions:
            letter_sections = True
            level4_no += 1
            level4_string = str('LETTER ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns defintions or sublaws -- letter sections present --> level 5
        if (re.search(fr'^([0-9]|[0-9][0-9])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level5_no==0)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and letter_sections and not in_definitions:
            in_alphabet = False
            level5_no += 1
            level5_string = str('sublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections present --> level 6
        if (re.search(r'\([a-t]\)', temp_text) or re.search(fr'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>0 and subsections and letter_sections and not in_definitions:
            in_alphabet = True
            level6_no += 1
            level6_string = str('subsublaw ' + str(level6_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'-\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and level5_no>=1 and subsections and letter_sections and not in_definitions:
            # bullets BEFORE lowercase letters --> level 6
            if level6_no==0:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 7
            elif level6_no>=1:
                level7_no += 1
                level7_string = str('subsubsublaw ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph_no = 0
                paragraph7 = False
                paragraph8 = False

        # returns sanctions -- letter sections present
        if (re.search(r'^(Sanction:|Penalty:)\s', temp_text) or re.search(r'\s(Sanction:|Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and subsections and letter_sections and not in_definitions:
            # same level as lowercase --> level 6
            if not in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 7
            elif in_alphabet:
                level7_no += 1
                level7_string = str('penalty/exception ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph_no = 0
                paragraph7 = False
                paragraph8 = False

        
        # returns defintions or sublaws -- letter sections NOT present --> level 4
        if (re.search(fr'^([0-9]|[0-9][0-9])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level4_no==0)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and subsections and not letter_sections and not in_definitions:
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns lowercase letters -- letter sections NOT present --> level 5
        if (re.search(r'\([a-t]\)', temp_text) or re.search(r'^(DEFINITION)', temp_text)) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and not letter_sections and not in_definitions:
            in_alphabet = True
            level5_no += 1
            level5_string = str('subsublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph_no = 0
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph8 = False

        # returns bullets -- letter sections present 
        if re.search(r'-\s', temp_text) and level1_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1 and subsections and not letter_sections and not in_definitions:
            # bullets BEFORE lowercase letters --> level 5
            if level5_no==0:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # bullets AFTER lowercase letters --> level 6
            elif level5_no>=1:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Penalty:)\s', temp_text) or re.search(r'\s(Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level2_no>=1 and level3_no>=1 and subsections and not letter_sections and not in_definitions:
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph_no = 0
                paragraph6 = False
                paragraph7 = False
                paragraph8 = False


        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|INSTRUCTIONS)).*', temp_text)) and len(temp_text)>=20 and level1_no>=1 and level2_no>=1 and real_law_no>=1 and not in_definitions:
            if (level2_no==0 or paragraph2) and level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph2 = True
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            elif (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0 and level8_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0 and level8_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7) and level8_no==0:
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0 and (level8_no==0 or paragraph8):
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0 and level8_no!=0:
                paragraph9 = True
                paragraph_no += 1
                level9_no += 1
                level9_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string][level8_no-1][level8_string].append({level9_string: []})
  

       

       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2000_2002 = (TEXTS_DIR / '2000-2002')
final_path_2000_2002 = (TREES_DIR / '2000-2002' / 'clean_')

for file_path in initial_path_2000_2002.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2000-2002', '')
        
        create_yamls_2000_2002(file, final_path_2000_2002, year)
        

In [ ]:
yaml_files_path_2000_2002 = (TREES_DIR / '2000-2002')

for file_path in yaml_files_path_2000_2002.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('tree', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_2000_2002_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    in_definitions = False
    in_directives = False
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})




    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_laws:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0


        # adds defintions
        if (re.search(r'^(?!(Rugby Union|\s|The following|These laws)).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})

    
        if re.search(r'^(BEFORE THE MATCH)', temp_text):
            in_definitions = False
            level0_no = 1
            level0_string = str('(main ruleset)')
       
        
        # main ruleset section
        '''if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0'''

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_definitions:
            in_laws = True
            in_definitions = False
            paragraph_no = 0
            real_law_no += 1
            level3_no += 1
            level3_string = str('LAW ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            level4_no = 0
        
        # returns defintions or sublaws -- letter sections NOT present --> level 4
        if (re.search(fr'^([0-9]|[0-9][0-9])\s[A-Z]', temp_text) or re.search(fr'^([0-9]|[0-9][0-9])\.\s[A-Z]', temp_text) or (re.search(fr'^(DEFINITION)', temp_text) and level4_no==0)) and level3_no>=1 and not in_definitions:
            paragraph_no = 0
            in_alphabet = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0

        # returns lowercase letters -- letter sections NOT present --> level 5
        if (re.search(r'\([a-t]\)', temp_text)) and level3_no>=1 and level4_no>=1 and not in_definitions:
            paragraph_no = 1
            in_alphabet = True
            level5_no += 1
            level5_string = str('subsublaw ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^-\s', temp_text) and level3_no>=1 and level4_no>=1 and not in_definitions:
            paragraph_no = 1
            # bullets BEFORE lowercase letters --> level 5
            if level5_no==0:
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            # bullets AFTER lowercase letters --> level 6
            elif level5_no>=1:
                level6_no += 1
                level6_string = str('subsubsublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})

        # returns sanctions -- letter sections NOT present
        if (re.search(r'^(Penalty:)\s', temp_text) or re.search(r'\s(Penalty:)\s', temp_text) or re.search(r'^(Exception:|Exception\s[0-9]:|EXCEPTION)', temp_text)) and level3_no>=1 and not in_definitions:
            paragraph_no = 1
            # same level as lowercase --> level 5
            if not in_alphabet:
                level5_no += 1
                level5_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                level6_no += 1
                level6_string = str('penalty/exception ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0

        
        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|[0-9]|\(|\s|NOTE|DEF|(P|F)\.K\.|[A-Z][A-Z]|[A-E]\.)).*', temp_text)) and level3_no>=1 and real_law_no>=1 and not in_definitions:
            if level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    level3_no += 1
                    level3_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level3_string: []})
            elif level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    level4_no += 1
                    level4_string = str('P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string].append({level4_string: []})
            elif level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})


       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_2000_2002 = (TEXTS_DIR / '2000-2002')
final_path_2000_2002_flat = (TREES_DIR / '2000-2002flat' / 'clean_')

for file_path in initial_path_2000_2002.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('2000-2002', '')
        
        create_yamls_2000_2002_flattened(file, final_path_2000_2002_flat, year)
        

In [ ]:
yaml_files_path_2000_2002_flat = (TREES_DIR / '2000-2002flat')

for file_path in yaml_files_path_2000_2002_flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1997-1998

The author's notes on this era's heading/depth pattern:

In [ ]:


# 1997-1998

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. DEFINITONS + LAW #. 
 2. A.
 3. (1) begins like or F.K. (1) or P.K. (1)
 4. (a) (not always first in line)
 5. bullets o or • 
 6. penalties:- + exceptions:-
 7. NOTE(S) (i), (ii), on same level as (a)
 8. IRB Ruling - or RFU Ruling - 
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# attempting to fix roman numeral i. 
def create_yamls_1997_1998(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    level8_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    notes3 = False
    notes4 = False
    in_bullet = False
    in_letter = False
    in_directives = False
    letter_sections = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False
    in_directives = False
    in_definitions = False
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_directives:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following|\s)).*', temp_text)) and in_definitions and not in_directives:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives and not in_definitions:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns letter sections --> level 3 if present
        if re.search(fr'^(A|B|C|D|E)\.\s', temp_text) and level2_no>=1 and not in_directives and not in_definitions:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            penalty4_letter = False
            penalty5_letter = False
            penalty4 = False
            penalty5 = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            notes = False
            letters_before4 = False
            bullets_before4 = False
            letters_before5 = False
            bullets_before5 = False
            ruling_possible = False
            ruling_real = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # determines if in notes section
        if re.search(r'(NOTE|NOTES)\s', temp_text) and not re.search(r'(INSTRUCTIONS)\s', temp_text) and not in_definitions and letter_sections:
            notes = True
            notes4 = False
            bullets_before4 = False
            bullets_before5 = False
            in_alphabet = False
            paragraph_no = 0
            # same level as numbers --> level 4
            if not in_alphabet and not in_bullet:
                notes4 = True
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 5
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                level6_no += 1
                level6_string = str('notes ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty6 or not re.search(r'^\(a\)\s', temp_text)) and not penalty5_letter and not penalty6_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    ruling_possible = False
                    in_alphabet = True
                    if level4_no==0 or letters_before4 or paragraph4:
                        letters_before4 = True
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6= False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty6 or not re.search(r'^\(a\)\s', temp_text)):
                    penalty5 = False
                    penalty6 = False
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 5
                    if not in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of lowercase --> subruling level 6
                    elif in_alphabet and not in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of bullet --> subruling level 7
                    elif in_bullet:
                        level7_no += 1
                        level7_string = str('ruling sublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                        paragraph7 = False
                # for notes
                elif notes:
                    if notes4:
                        level6_no += 1
                        level6_string = str('notes sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                    elif not notes4:
                        level7_no += 1
                        level7_string = str('notes sublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                        level8_no = 0
                        paragraph7 = False
                # for penalty following numbers
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in peanlty
                        if not penalty5_roman:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in peanlty
                        else: 
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        penalty5_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty6:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty6_letter and re.search(r'\(a\)\s', temp_text)) or penalty6_letter:
                        # no roman numerals in penalty
                        if not penalty6_roman:
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        # roman numerals in penalty
                        else:
                            level8_no += 1
                            level8_string = str('penalty sublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
                            level9_no = 0
                            paragraph8 = False
                        penalty6_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty5_letter and not penalty6_letter:
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]: 
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6=False  
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False  

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text) or re.search(r':\s\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            # roman numerals in notes
            if notes:
                if level4_no==0 or penalty5 or notes4:
                    notes4 = True
                    level5_no += 1
                    notes_no += 1
                    level5_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    notes_no += 1
                    level6_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
            # in penalty following number sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = str('penalty sublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
            # in penalty following letter sublaw
            elif penalty6:
                penalty6_roman = True
                penalty6_letter = False
                level7_no += 1
                level7_string = str('penalty sublaw ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False

        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and not in_definitions:
            notes = False
            penalty5_roman = False
            penalty5_letter = False
            penalty6_roman = False
            penalty6_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 0
            # same level as lowercase --> level 5
            if not in_alphabet:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    paragraph5 = False
                    paragraph7 = False
                    paragraph6 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                else: 
                    paragraph6 = False
                    paragraph7 = False
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 4
            if level4_no==0 or bullets_before4 or paragraph4:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 5
            elif level5_no==0 or bullets_before5:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    paragraph6 = False
                    paragraph7 = False
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_letter and not penalty6_roman:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # if only penalty letter
                    elif not penalty6_roman and penalty6_letter:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0
                    # following both roman numerals and lowercase
                    elif penalty6_letter and penalty6_roman:
                        if level8_no>1:
                            level9_no += 1
                            level9_string = str('penalty subsublaw ' + str(level9_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string][level8_no-1][level8_string].append({level9_string:[]})

        # returns rulings 
        if (re.search(r'^(RFU|IRFB)\sRuling\s-', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 0
            # same level as numbers --> level 6
            if not in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of lowercase --> level 5
            elif in_alphabet and not in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level7_no += 1
                    level7_string = str('ruling ' + str(level7_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                    level8_no = 0
                    paragraph7 = False






        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)\s', temp_text) and not re.search(r'^(INSTRUCTIONS)\s', temp_text)  and not letter_sections and not in_directives and real_law_no>=1 and not in_definitions:
            notes = True
            notes3 = False
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 0
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                notes3 = True
                level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 4
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\([a-r]\)\s', temp_text)) and (not penalty5 or not re.search(r'^\([a-r]\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    ruling_possible = False
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3 or paragraph3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                # for notes
                elif notes:
                    if notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    elif not notes3:
                        level6_no += 1
                        level6_string = str('notes sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                            level7_no = 0
                            paragraph5 = False
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in peanlty
                        else: 
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in penalty
                        else:
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty4_letter and not penalty5_letter:
                paragraph_no = 0
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string]:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        v

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'^\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text) or re.search(r':\s\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                else:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level5_no += 1
                    notes_no += 1
                    level5_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
            # in penalty following number sublaw
            elif penalty4:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4_roman = True
                penalty4_letter = False
                level5_no += 1
                level5_string = str('penalty sublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                paragraph6 = False
                paragraph7 = False
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = str('penalty sublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                else:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3 or paragraph3:
                bullets_before3 = True
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            paragraph5 = False
                            paragraph6 = False
                            paragraph7 = False
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                            level6_no = 0
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_letter:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            paragraph7 = 0
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 0
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False



        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|INSTRUCTIONS|NOTE|Penalty|Penalties|Exception|IRFB|RFU)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and not in_definitions:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7):
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
  

           
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1997_1998 = (TEXTS_DIR / '1997-1998')
final_path_1997_1998 = (TREES_DIR / '1997-1998' / 'clean_')

for file_path in initial_path_1997_1998.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('1997-1998', '')
        
        create_yamls_1997_1998(file, final_path_1997_1998, year)
        

In [ ]:
yaml_files_path_1997_1998 = (TREES_DIR / '1997-1998')

for file_path in yaml_files_path_1997_1998.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
# adding rulings to notes
def create_yamls_1997_1998_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    in_definitions = False
    in_laws = False


    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)):
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following|\s)).*', temp_text)) and in_definitions and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)):
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)):
            in_laws = True
            in_definitions = False
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            ruling_possible = False
            ruling_real = False
            paragraph_no = 1

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)\s', temp_text) and not in_definitions:
            notes = True
            notes3 = False
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    penalty4 = False 
                    penalty5 = False 
                    ruling_possible = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                # for notes
                elif notes:
                    if notes3:
                        level4_no += 1
                        level4_string = str('notes sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                    elif not notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                # letters as sublaws of number sublaws
                if not notes and not penalty4_letter and not penalty5_letter:
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions:
            paragraph_no = 1
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    notes3 = True
                    level3_no += 1
                    notes_no += 1
                    level3_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                else:
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
            # in penalty following number sublaw
            elif penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_definitions:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*)\s', temp_text) and level2_no>=1 and not letter_sections and not in_definitions:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0


        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|\s|RFU|IRFB|NOTE|DEF|(P|F)\.K\.)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and len(temp_text) > 40 and not in_definitions:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})


        
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1997_1998 = (TEXTS_DIR / '1997-1998')
final_path_1997_1998_flat = (TREES_DIR / '1997-1998flat' / 'clean_')

for file_path in initial_path_1997_1998.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        year = year.replace('1997-1998', '')
        
        create_yamls_1997_1998_flattened(file, final_path_1997_1998_flat, year)

In [ ]:
yaml_files_path_1997_1998flat = (TREES_DIR / '1997-1998flat')

for file_path in yaml_files_path_1997_1998flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1985-1995

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1985-1995

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. DEFINITONS + LAW #. + INTERNATIONAL RUGBY FOOTBALL BOARD DIRECTIVES
 2. A.
 3. (1) begins like or F.K. (1) or P.K. (1)
 4. (a) (not always first in line)
 5. bullets o or • 
 6. penalties:- + exceptions:-
 7. NOTE(S) (i), (ii), on same level as (a)
 8. IRB Ruling - or RFU Ruling - 
 -- irfb ruling sections:
    - Dates
        - Questions + Answers
        - Law #
        - other headings 
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# attempting to fix roman numeral i. 
def create_yamls_1985_1995(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    level8_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    notes3 = False
    notes4 = False
    in_bullet = False
    in_letter = False
    in_directives = False
    letter_sections = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False
    in_directives = False
    in_definitions = False
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_directives:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following|\s)).*', temp_text)) and in_definitions and not in_directives:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives and not in_definitions:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns letter sections --> level 3 if present
        if re.search(fr'^(A|B|C|D|E)\.\s', temp_text) and level2_no>=1 and not in_directives and not in_definitions:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            penalty4_letter = False
            penalty5_letter = False
            penalty4 = False
            penalty5 = False
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            notes = False
            letters_before4 = False
            bullets_before4 = False
            letters_before5 = False
            bullets_before5 = False
            ruling_possible = False
            ruling_real = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # determines if in notes section
        if re.search(r'(NOTE|NOTES)\s', temp_text) and not re.search(r'(INSTRUCTIONS)\s', temp_text) and not in_definitions and letter_sections:
            notes = True
            notes4 = False
            bullets_before4 = False
            bullets_before5 = False
            in_alphabet = False
            paragraph_no = 0
            # same level as numbers --> level 4
            if not in_alphabet and not in_bullet:
                notes4 = True
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 5
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                level6_no += 1
                level6_string = str('notes ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty6 or not re.search(r'^\(a\)\s', temp_text)) and not penalty5_letter and not penalty6_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    ruling_possible = False
                    in_alphabet = True
                    if level4_no==0 or letters_before4 or paragraph4:
                        letters_before4 = True
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6= False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty6 or not re.search(r'^\(a\)\s', temp_text)):
                    penalty5 = False
                    penalty6 = False
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 5
                    if not in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of lowercase --> subruling level 6
                    elif in_alphabet and not in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of bullet --> subruling level 7
                    elif in_bullet:
                        level7_no += 1
                        level7_string = str('ruling sublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                        paragraph7 = False
                # for notes
                elif notes:
                    if notes4:
                        level6_no += 1
                        level6_string = str('notes sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                    elif not notes4:
                        level7_no += 1
                        level7_string = str('notes sublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                        level8_no = 0
                        paragraph7 = False
                # for penalty following numbers
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in peanlty
                        if not penalty5_roman:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in peanlty
                        else: 
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        penalty5_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty6:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty6_letter and re.search(r'\(a\)\s', temp_text)) or penalty6_letter:
                        # no roman numerals in penalty
                        if not penalty6_roman:
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        # roman numerals in penalty
                        else:
                            level8_no += 1
                            level8_string = str('penalty sublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
                            level9_no = 0
                            paragraph8 = False
                        penalty6_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty5_letter and not penalty6_letter:
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]: 
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6=False  
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False  

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text) or re.search(r':\s\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            # roman numerals in notes
            if notes:
                if level4_no==0 or penalty5 or notes4:
                    notes4 = True
                    level5_no += 1
                    notes_no += 1
                    level5_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    notes_no += 1
                    level6_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
            # in penalty following number sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = str('penalty sublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
            # in penalty following letter sublaw
            elif penalty6:
                penalty6_roman = True
                penalty6_letter = False
                level7_no += 1
                level7_string = str('penalty sublaw ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False

        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and not in_definitions:
            notes = False
            penalty5_roman = False
            penalty5_letter = False
            penalty6_roman = False
            penalty6_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 0
            # same level as lowercase --> level 5
            if not in_alphabet:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    paragraph5 = False
                    paragraph7 = False
                    paragraph6 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                else: 
                    paragraph6 = False
                    paragraph7 = False
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 4
            if level4_no==0 or bullets_before4 or paragraph4:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 5
            elif level5_no==0 or bullets_before5:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    paragraph6 = False
                    paragraph7 = False
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_letter and not penalty6_roman:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # if only penalty letter
                    elif not penalty6_roman and penalty6_letter:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0
                    # following both roman numerals and lowercase
                    elif penalty6_letter and penalty6_roman:
                        if level8_no>1:
                            level9_no += 1
                            level9_string = str('penalty subsublaw ' + str(level9_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string][level8_no-1][level8_string].append({level9_string:[]})

        # returns rulings 
        if (re.search(r'^(RFU|IRFB)\sRuling\s-', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 0
            # same level as numbers --> level 6
            if not in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of lowercase --> level 5
            elif in_alphabet and not in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level7_no += 1
                    level7_string = str('ruling ' + str(level7_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                    level8_no = 0
                    paragraph7 = False






        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)\s', temp_text) and not re.search(r'^(INSTRUCTIONS)\s', temp_text)  and not letter_sections and not in_directives and real_law_no>=1 and not in_definitions:
            notes = True
            notes3 = False
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 0
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                notes3 = True
                level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of lowercase --> level 4
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\([a-r]\)\s', temp_text)) and (not penalty5 or not re.search(r'^\([a-r]\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    ruling_possible = False
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3 or paragraph3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                # for notes
                elif notes:
                    if notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    elif not notes3:
                        level6_no += 1
                        level6_string = str('notes sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                        level7_no = 0
                        paragraph6 = False
                        paragraph7 = False
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                            level7_no = 0
                            paragraph5 = False
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in peanlty
                        else: 
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                            paragraph6 = False
                            paragraph7 = False
                        # roman numerals in penalty
                        else:
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                            paragraph7 = False
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty4_letter and not penalty5_letter:
                paragraph_no = 0
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string]:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        v

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'^\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text) or re.search(r':\s\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                else:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level5_no += 1
                    notes_no += 1
                    level5_string = str('notes point ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
            # in penalty following number sublaw
            elif penalty4:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4_roman = True
                penalty4_letter = False
                level5_no += 1
                level5_string = str('penalty sublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                paragraph6 = False
                paragraph7 = False
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = str('penalty sublaw ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                else:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3 or paragraph3:
                bullets_before3 = True
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            paragraph5 = False
                            paragraph6 = False
                            paragraph7 = False
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                            level6_no = 0
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_letter:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            paragraph7 = 0
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty subsublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                            level9_no = 0

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 0
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    paragraph6 = False
                    paragraph7 = False



        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|INSTRUCTIONS|NOTE|Penalty|Penalties|Exception|IRFB|RFU)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and not in_definitions:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7):
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
  



        
        # IRB Directives
        if re.search('^(INTERNATIONAL RUGBY FOOTBALL BOARD DIRECTIVES)', temp_text)  and not in_definitions:
            in_laws = False
            in_directives = True
            real_law_no += 1
            level0_no = 3
            level0_string = str('IRB DIRECTIVES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            continue

        if re.search(r'^(January|February|March|April|May|June|July|August|September|October|November|December)', temp_text, re.IGNORECASE) and in_directives and not in_definitions:
            level2_no += 1
            level2_string = str('heading ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0

        if re.search(r'^(?!([0-9]|Law|NOTE|January|February|March|April|May|June|July|August|September|October|November|December|Amendments to the Laws of the Game)).*', temp_text, re.IGNORECASE) and in_directives and len(temp_text)>=40 and not in_definitions:
            if level4_no==0:
                level3_no += 1
                level3_string = str('P' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            else:
                level5_no += 1
                level5_string = str('P' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
        
        if re.search(r'^([0-9]\.|[0-9][0-9]\.|NOTE)', temp_text) and in_directives and level3_no>=1 and not in_definitions:
            level4_no += 1
            level4_string = str('point ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0


           
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1985_1995 = (TEXTS_DIR / '1985-1995')
final_path_1985_1995 = (TREES_DIR / '1985-1995' / 'clean_')

for file_path in initial_path_1985_1995.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1985_1995(file, final_path_1985_1995, year)

In [ ]:
yaml_files_path_1985_1995 = (TREES_DIR / '1985-1995')

for file_path in yaml_files_path_1985_1995.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
# adding rulings to notes
def create_yamls_1985_1995_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_alphabet = False
    in_bullet = False
    notes = False
    in_directives = False
    in_definitions = False
    in_laws = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_directives:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following|\s)).*', temp_text)) and in_definitions and not in_directives and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            in_laws = True
            in_definitions = False
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            notes3 = False
            notes4 = False
            notes5 = False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            ruling_possible = False
            ruling_real = False
            paragraph_no = 1

        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            notes = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            ruling_possible = False
            ruling_real = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            paragraph_no = 1

        # determines if in notes section
        if re.search(r'^(NOTE|NOTES)', temp_text) and not in_directives and real_law_no>=1 and not in_definitions:
            notes = True
            bullets_before3 = False
            bullets_before4 = False
            in_alphabet = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                notes3 = True
                '''level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0'''
            # sublevel of lowercase --> level 4
            elif (in_alphabet and not in_bullet) or (in_bullet and not in_alphabet):
                notes4 = True
                '''level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0'''
            # sublevel of bullet
            elif in_bullet:
                notes5 = True
                '''level5_no += 1
                level5_string = str('notes ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0'''


        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if not notes and (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter and (not ruling_real or (ruling_possible and not re.search(r'\(a\)\s', temp_text))):
                    penalty4 = False
                    penalty5 = False
                    ruling_possible = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        penalty5=0
                # for rulings 
                elif ((ruling_possible and re.search(r'\(a\)\s', temp_text)) or ruling_real) and not notes and not penalty4 and not penalty5:
                    ruling_real = True
                    # ruling is same level as numbers, so subruling --> level 4
                    if not in_alphabet and not in_bullet:
                        level4_no += 1
                        level4_string = str('ruling sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                    # sublevel of lowercase --> subruling level 5
                    elif in_alphabet and not in_bullet:
                        level5_no += 1
                        level5_string = str('ruling sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level5_no = 0
                        level6_no = 0
                    # sublevel of bullet --> subruling level 6
                    elif in_bullet:
                        level6_no += 1
                        level6_string = str('ruling sublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                # for notes
                elif notes:
                    if notes3:
                        level4_no += 1
                        level4_string = str('notes sublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                    elif not notes3:
                        level5_no += 1
                        level5_string = str('notes sublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                                level5_no = 0
                                level6_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and not notes and not penalty4_letter and not penalty5_letter:
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if ({'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string]):
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                else:
                    if ({'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]):
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        penalty5=0

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections NOT present 
        notes_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        if (re.search(r'\((i|ii|iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            paragraph_no = 1
            # roman numerals in notes
            if notes:
                if level3_no==0 or penalty4 or notes3:
                    notes3 = True
                    level3_no += 1
                    notes_no += 1
                    level3_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                    level5_no = 0
                    level6_no = 0
                else:
                    level4_no += 1
                    notes_no += 1
                    level4_string = str('notes ' +str(notes_labels[notes_no-1]))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
            # in penalty following number sublaw
            elif penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # in penalty following letter sublaw
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions:
            notes = False
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            ruling_real = False
            ruling_possible = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    penalty4 = False
                    penalty5 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_letter:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # returns rulings -- letter sections NOT present 
        if (re.search(r'^(RFU|IRFB)\sRuling', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            ruling_possible = True
            penalty4 = False
            penalty5 = False
            paragraph_no = 1
            # same level as numbers --> level 3
            if not in_alphabet and not in_bullet:
                if not notes:
                    level3_no += 1
                    level3_string = str('ruling ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                    level5_no = 0
                    level6_no = 0
                else:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
            # sublevel of lowercase --> level 4
            elif in_alphabet and not in_bullet:
                if not notes:
                    level4_no += 1
                    level4_string = str('ruling ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                else:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
            # sublevel of bullet
            elif in_bullet:
                if not notes:
                    level5_no += 1
                    level5_string = str('ruling ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('ruling ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|[0-9]|\s|RFU|IRFB|NOTE|INTER)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and not in_definitions:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})

    

        # IRB Directives
        if re.search('^(INTERNATIONAL RUGBY FOOTBALL BOARD DIRECTIVES)', temp_text):
            in_laws = False
            in_directives = True
            real_law_no += 1
            level0_no = 3
            level0_string = str('IRB DIRECTIVES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            continue

        if re.search(r'^(?!([0-9]|NOTE|January|February|March|April|May|June|July|August|September|October|November|December|Amendments to the Laws of the Game)).*', temp_text, re.IGNORECASE) and in_directives and len(temp_text)>=40:
            level2_no += 1
            level2_string = str('P' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            continue
        
        if re.search(r'^([0-9]\.|[0-9][0-9]\.|NOTE)', temp_text) and in_directives and level2_no>=1:
            level3_no += 1
            level3_string = str('point ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            

    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1985_1995 = (TEXTS_DIR / '1985-1995')
final_path_1985_1995flat = (TREES_DIR / '1985-1995flat' / 'clean_')

for file_path in initial_path_1985_1995.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1985_1995_flattened(file, final_path_1985_1995flat, year)

In [ ]:
yaml_files_path_1985_1995flat = (TREES_DIR / '1985-1995flat')

for file_path in yaml_files_path_1985_1995flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1969-1974

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1969-1974

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. DEFINITONS + LAW #. + NOTES (did not make it into text files)
 2. LAW # A. or A. 
 3. (1) begins like or F.K. (1) or P.K. (1)
 4. (a) (not always first in line)
 5. bullets o or • 
 6. Penalty: + Exception(s):(-) with (i), (a), bullet
 
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:

def create_yamls_1969_1974(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    in_directives = False
    in_definitions = False
    letter_sections = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False
    paragraph7 = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})

    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_directives:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following)).*', temp_text)) and in_definitions and not in_directives:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       


        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            in_definitions = False
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            paragraph3 = True
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns letter sections --> level 3 if present
        if (re.search(fr'^(A|B|C|D|E)\.\s', temp_text) or re.search(fr'^(LAW)\s([0-9]|[0-9][0-9])\s(A|B|C|D|E)\.\s', temp_text)) and level2_no>=1 and not in_directives and not in_definitions:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            penalty5_letter = False
            penalty6_letter = False
            penalty5 = False
            penalty6 = False
            paragraph3 = False
            paragraph4 = True
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            letters_before4 = False
            bullets_before4 = False
            bullets_before5 = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = True
            paragraph6 = False
            paragraph7 = False
            paragraph_no = 0

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 0
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty6 or not re.search(r'^\(a\)\s', temp_text))and not penalty5_letter and not penalty6_letter:
                    in_alphabet = True
                    if level4_no==0 or letters_before4 or paragraph4:
                        letters_before4 = True
                        paragraph3 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                    else:
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6=False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for penalty following numbers
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in peanlty
                        if not penalty5_roman:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        # roman numerals in peanlty
                        else: 
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                        penalty5_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty6:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty6_letter and re.search(r'\(a\)\s', temp_text)) or penalty6_letter:
                        # no roman numerals in penalty
                        if not penalty6_roman:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                        # roman numerals in penalty
                        else:
                            paragraph8 = False
                            level8_no += 1
                            level8_string = str('penalty sublaw ' + str(level8_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})
                            level9_no = 0
                        penalty6_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]:
                # letters as sublaws of number sublaws
                if not penalty5_letter and not penalty6_letter:
                    in_alphabet = True
                    if level4_no==0 or letters_before4:
                        letters_before4 = True
                        paragraph3 = False
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        penalty6=False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False   


        # returns roman numerals -- letter sections present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            if penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 2
                level6_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
            elif penalty6:
                penalty6_roman = True
                penalty6_letter = False
                level7_no += 2
                level7_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False
            else: 
                level6_no += 2
                level6_string = str('ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0 
                paragraph6 = False
                paragraph7 = False
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            if penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = str('penalty ' + str(roman_labels[level6_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
            elif penalty6:
                penalty6_roman = True
                penalty6_letter = False
                level7_no += 1
                level7_string = str('penalty ' + str(roman_labels[level7_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
                paragraph7 = False
            else: 
                level6_no += 1
                level6_string = roman_labels[level6_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0 
                paragraph6 = False
                paragraph7 = False
            
        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:|Penalties:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives and not in_definitions:
            penalty5_roman = False
            penalty5_letter = False
            penalty6_roman = False
            penalty6_letter = False
            paragraph_no = 0
            # same level as lowercase --> level 5
            if not in_alphabet:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                else: 
                    paragraph6 = False
                    paragraph7 = False
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 4
            if (level4_no==0 or bullets_before4 or paragraph4) and not penalty5 and not penalty6:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 5
            elif (level5_no==0 or bullets_before5) and not penalty5 and not penalty6:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    paragraph6 = False
                    paragraph7 = False
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        #if level5_no>1:
                        paragraph6 = False
                        paragraph7 = False
                        level6_no += 1
                        level6_string = str('penalty subsublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        #if level6_no>1:
                        paragraph7 = False
                        level7_no += 1
                        level7_string = str('penalty subsublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        #if level7_no>1:
                        paragraph8 = False
                        level8_no += 1
                        level8_string = str('penalty subsublaw ' + str(level8_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                        level9_no = 0
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_letter and not penalty6_roman:
                        #if level6_no>1:
                        paragraph7 = False
                        level7_no += 1
                        level7_string = str('penalty subsublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                    # if only penalty letter
                    elif not penalty6_roman and penalty6_letter:
                        #if level7_no>1:
                        paragraph8 = False
                        level8_no += 1
                        level8_string = str('penalty subsublaw ' + str(level8_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                        level9_no = 0
                    # following both roman numerals and lowercase
                    elif penalty6_letter and penalty6_roman:
                        #if level8_no>1:
                        paragraph9 = False
                        level9_no += 1
                        level9_string = str('penalty subsublaw ' + str(level9_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string][level8_no-1][level8_string].append({level9_string:[]})


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            paragraph3 = False
            paragraph4 = True
            paragraph5 = False
            paragraph6 = False
            paragraph7 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph_no = 0

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter:
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3 or paragraph3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=0
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            paragraph5 = False
                            paragraph6 = False
                            paragraph7 = False
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                            level7_no = 0
                        # roman numerals in peanlty
                        else: 
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            paragraph6 = False
                            paragraph7 = False
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        # roman numerals in penalty
                        else:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty sublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
                            level8_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                # letters as sublaws of number sublaws
                if not penalty4_letter and not penalty5_letter:
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=0
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False

        # returns roman numerals -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            if penalty4:
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4_roman = True
                penalty4_letter = False
                level5_no += 2
                level5_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            elif penalty5:
                paragraph6 = False
                paragraph7 = False
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 2
                level6_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                level5_no += 2
                level5_string = str('ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0 
                level7_no = 0
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not letter_sections and not in_directives and not in_definitions:
            paragraph_no = 0
            if penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level5_no += 1
                level5_string = roman_labels[level5_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level6_no += 1
                level6_string = roman_labels[level6_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
                paragraph7 = False
            else: 
                level5_no += 1
                level5_string = roman_labels[level5_no-1]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0 
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False

        # returns penalties or exceptions  -- letter sections NOT present 
        if (re.search(r'^(Penalty:|Penalties:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives and not in_definitions:
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            paragraph_no = 0
            # same level as lowercase --> level 4
            if not in_alphabet:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                paragraph4 = False
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                else:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 3
            if (level3_no==0 or bullets_before3 or paragraph3) and not penalty4 and not penalty5:
                bullets_before3 = True
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 4
            elif (level4_no==0 or bullets_before4) and not penalty4 and not penalty5:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                paragraph7 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    paragraph5 = False
                    paragraph6 = False
                    paragraph7 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        #if level4_no>1:
                        paragraph5 = False
                        paragraph6 = False
                        paragraph7 = False
                        level5_no += 1
                        level5_string = str('penalty subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                        level6_no = 0
                        level7_no = 0
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_letter:
                        #if level5_no>1:
                        paragraph6 = False
                        paragraph7 = False
                        level6_no += 1
                        level6_string = str('penalty subsublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        #if level6_no>1:
                        paragraph7 = False
                        level7_no += 1
                        level7_string = str('penalty subsublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        #if level5_no>1:
                        paragraph6 = False
                        paragraph7 = False
                        level6_no += 1
                        level6_string = str('penalty subsublaw ' + str(level6_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                        level7_no = 0
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_letter:
                        #if level6_no>1:
                        paragraph7 = False
                        level7_no += 1
                        level7_string = str('penalty subsublaw ' + str(level7_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                        level8_no = 0
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        #if level7_no>1:
                        paragraph8 = False
                        level8_no += 1
                        level8_string = str('penalty subsublaw ' + str(level8_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string:[]})
                        level9_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(LAW|A\.|B\.|C\.|D\.|E\.|\s|INSTRUCTIONS|\(i\))).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives and not in_definitions:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0 and level7_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6) and level7_no==0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and (level7_no==0 or paragraph7):
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph8 = True
                paragraph_no += 1
                level8_no += 1
                level8_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-1][level7_string].append({level8_string: []})




        # NOTES
        if re.search('^(FOREWORD)', temp_text):
            in_directives = True
            level0_no = 3
            level0_string = str('IRFB DIRECTIVES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            heading_no = 0
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law|FOREWORD|(For your))', temp_text)) and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0

        if (re.search(r'^(?!((NOTES)|(LAW|Law)|(For your)|\(|\s)).*', temp_text)) and level2_no>=1 and level0_no>=1 and in_directives:
            level3_no += 1
            level3_string = str('P ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
        
        if re.search(r'^\(([1-9]|[1-9][0-9])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1:
            if level3_no>=1:
                level4_no += 1
                level4_string = str('point ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
            else:
                level3_no += 1
                level3_string = str('point ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            
        if re.search(r'^\(([a-z])\)', temp_text) and in_directives  and level2_no>=1 and level3_no>=1 and level0_no>=1:
            if level4_no>=1:
                level5_no += 1
                level5_string = str('subpoint ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
            else:
                level4_no += 1
                level4_string = str('subpoint ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0

       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1969_1974 = (TEXTS_DIR / '1969-1974')
final_path_1969_1974 = (TREES_DIR / '1969-1974' / 'clean_')

for file_path in initial_path_1969_1974.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')

        create_yamls_1969_1974(file, final_path_1969_1974, year)

In [ ]:
yaml_files_path_1969_1974 = (TREES_DIR / '1969-1974')

for file_path in yaml_files_path_1969_1974.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:

def create_yamls_1969_1974_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    level7_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    notes = False
    in_directives = False
    letter_sections = False
    in_definitions = False
    in_laws = False


    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

        # defintions section
        if (re.search(r'^(DEFINITIONS)', temp_text)) and not in_directives:
            in_definitions = True
            level0_no = 2
            level0_string = str('DEFINITIONS')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level3_no = 0

        # adds defintions
        if (re.search(r'^(?!(The following)).*', temp_text)) and in_definitions and not in_directives and not in_laws:
            level2_no += 1
            level2_string = str('definitions ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
       



        # main ruleset section
        if (not in_definitions or re.search(r'^(LAW)\s1\.', temp_text)) and not in_directives:
            level0_no = 1
            level0_string = str('(main ruleset)')
            in_definitions = False
            if re.search(r'^(LAW)\s1\.', temp_text):
                level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            in_laws = True
            in_definitions = False
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            notes_no = 0
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            notes=False
            in_alphabet = False
            in_bullet = False
            penalty4_letter = False
            penalty5_letter = False
            penalty6_letter = False
            paragraph_no = 0


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not in_directives and not in_definitions:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives and not in_definitions:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)) and not penalty4_letter and not penalty5_letter:
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
                # for penalty following numbers
                elif penalty4:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty4_letter and re.search(r'\(a\)\s', temp_text)) or penalty4_letter:
                        # no roman numerals in peanlty
                        if not penalty4_roman:
                            if penalty4_letter:
                                level4_no += 1
                                level4_string = str('penalty sublaw ' + str(level4_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                                level4_no = 0
                        # roman numerals in peanlty
                        else: 
                            level5_no += 1
                            level5_string = str('penalty sublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                            level6_no = 0
                        penalty4_letter = True
                # for penalty following lowercase letter sublaw
                elif penalty5:
                    # makes sure to only enter penalty letter sublist when beginning with a
                    if (not penalty5_letter and re.search(r'\(a\)\s', temp_text)) or penalty5_letter:
                        # no roman numerals in penalty
                        if not penalty5_roman:
                            if not penalty5_letter:
                                level5_no += 1
                                level5_string = str('penalty sublaw ' + str(level5_no))
                                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                                level6_no = 0
                        # roman numerals in penalty
                        else:
                            level6_no += 1
                            level6_string = str('penalty sublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                            level7_no = 0
                        penalty5_letter = True
            elif (re.match(r'\(i\)\s', temp_text) or re.search(r'^\([0-9]\)\s\(i\)\s', temp_text)) and {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                # letters as sublaws of number sublaws
                if not penalty4_letter and not penalty5_letter:
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        # returns roman numerals -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and not in_definitions:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            else: 
                level5_no += 2
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                level6_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and not in_directives and not in_definitions:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                penalty4_letter = False
                level4_no += 1
                level4_string = roman_labels[level4_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                penalty5_letter = False
                level5_no += 1
                level5_string = roman_labels[level5_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                level5_no += 1
                level5_string = roman_labels[level5_no]
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:|Penalties:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives and not in_definitions:
            penalty4_roman = False
            penalty4_letter = False
            penalty5_roman = False
            penalty5_letter = False
            paragraph_no = 1
            # same level as lowercase --> level 4
            if not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0

        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not in_directives and not in_definitions:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_letter and not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif not penalty4_roman and penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty4_letter and penalty4_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_letter and not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif not penalty5_roman and penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty5_letter and penalty5_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalt|o|•|\*|-|\(|[0-9]|\s|NOTE|INTER|A\.|B\.|C\.|D\.|E\.)).*', temp_text)) and not in_directives and in_laws:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1  
                if paragraph_no >=2:  
                    level2_no += 1
                    level2_string = str('LAW ' + str(real_law_no) + ' P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0 and level7_no==0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no==0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0 and level7_no!=0:
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})



        # NOTES
        if re.search('^(FOREWORD)', temp_text):
            in_directives = True
            level0_no = 3
            level0_string = str('IRFB DIRECTIVES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            heading_no = 0
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law|FOREWORD|(For your))', temp_text)) and in_directives:
            heading_no += 1
            continue

        if (re.search(r'^(?!((NOTES)|(LAW|Law)|(For your)|\(|\s)).*', temp_text)) and level0_no>=1 and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(heading_no) + ' P' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
        
        if re.search(r'^\(([1-9]|[1-9][0-9])\)', temp_text) and in_directives and level0_no>=1:
            if level2_no>=1:
                level3_no += 1
                level3_string = str('point ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            else:
                level2_no += 1
                level2_string = str('heading ' + str(heading_no) + ' point ' + str(level2_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
                level3_no = 0
            
        if re.search(r'^\(([a-z])\)', temp_text) and in_directives and level3_no>=1 and level4_no>=1:
            if level3_no>=1:
                level4_no += 1
                level4_string = str('subpoint ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
            else:
                level3_no += 1
                level3_string = str('subpoint ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0


        
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1969_1974 = (TEXTS_DIR / '1969-1974')
final_path_1969_1974flat = (TREES_DIR / '1969-1974flat' / 'clean_')

for file_path in initial_path_1969_1974.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')

        create_yamls_1969_1974_flattened(file, final_path_1969_1974flat, year)

In [ ]:
yaml_files_path_1969_1974flat = (TREES_DIR / '1969-1974flat')

for file_path in yaml_files_path_1969_1974flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1954-1959

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1954-1959

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. LAW 1 + PART I + Notes -- remove parts in flattened
 2. LAW #.s
 3. (A), (B) -- not removable in flattened since have text
 4. (1), (2), (3)
 5. (a), (b) -- sometimes skips (1)
 6. (i), (ii) or 1., 2. 
 7. penalty: (i), (ii) or (1), (2)
 
exception: law 34 has (a) -> (1) -- changed in text file to 1.
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# removing penalty letters
def create_yamls_1954_1959(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    letter_sections = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

        # returns parts --> level 1
        if (re.search(r'^(Part)\s(I|II|III)\.', temp_text)) and not in_directives:
            level1_no += 1
            level1_string = str('part ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0

        # returns laws --> level 2
        if (re.search(r'^(?!LAW\s1\.)(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            letter_no = 0
            letter_sections = False
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            penalty6 = False
            penalty4_roman = False
            penalty5_roman = False
            penalty6_roman = False
            paragraph3 = True
            paragraph4 = False
            paragraph5 = False
            paragraph6 = False
            paragraph_no = 0

        # returns letter sections --> level 3 if present
        if re.search(fr'^\((A|B|C|D|E)\)\s', temp_text) and level2_no>=1 and not in_directives:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            penalty4 = False
            penalty5 = False
            paragraph3 = False
            paragraph4 = True
            paragraph5 = False
            paragraph6 = False
            paragraph_no = 0

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            letters_before4 = False
            bullets_before4 = False
            letters_before5 = False
            bullets_before5 = False
            ruling_possible = False
            ruling_real = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            level6_no = 0
            level7_no = 0
            level8_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = True
            paragraph6 = False
            paragraph_no = 0

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 0
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    letters_before4 = True
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                    level8_no = 0
                    bullets_before = False
                    penalty5 = False
                    penalty6 = False
                elif level5_no==0 or paragraph4:
                    level5_no += 1
                    level5_string = str('subsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                    level6_no = 0
                    level7_no = 0
                    level8_no = 0
                    paragraph5 = False
                    paragraph6 = False
                else:
                    level6_no += 1
                    level6_string = str('subsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
                    level7_no = 0
                    level8_no = 0
                    paragraph6 = False
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        level8_no = 0
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]: 
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        level8_no = 0
                        paragraph5 = False
                        paragraph6 = False
   

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph5 = False
            paragraph_no = 0
            if penalty5:
                penalty5_roman = True
                roman_no = 2
                level6_no += 2
                level6_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
            elif penalty6:
                paragraph7 = False
                penalty6_roman = True
                roman_no = 2
                level7_no += 2
                level7_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string][level7_no-2]['penalty sublaw i. '].append({'P':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
            else: 
                if letters_before4:
                    roman_no = 2
                    level5_no += 2
                    level5_string = str('ii. ')
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['i. '].append({'P':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 
                    level7_no = 0
                    level8_no = 0
                    paragraph5 = False
                    paragraph6 = False
                else:
                    roman_no = 2
                    level6_no += 2
                    level6_string = str('ii. ')
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['i. '].append({'P':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0 
                    level8_no = 0
                    paragraph6 = False
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph_no = 0
            if penalty5:
                penalty5_roman = True
                roman_no += 1
                level6_no += 1
                level6_string = str('penalty' + str(roman_labels[roman_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
                paragraph6 = False
            elif penalty6:
                penalty6_roman = True
                roman_no += 1
                level7_no += 1
                level7_string = str('penalty ' + str(roman_labels[roman_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
            else: 
                if letters_before4:
                    roman_no += 1
                    level5_no += 1
                    level5_string = roman_labels[roman_no-1]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    level8_no = 0
                    paragraph5 = False
                    paragraph6 = False
                else:
                    roman_no += 1
                    level6_no += 1
                    level6_string = roman_labels[roman_no-1]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    level8_no = 0 
                    paragraph6 = False

        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            penalty5_roman = False
            penalty6_roman = False
            paragraph_no = 0
            # no sub-points --> level 4
            if level4_no==0:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
                level8_no = 0
            # same level as lowercase --> level 5
            elif not in_alphabet:
                paragraph5 = False
                paragraph6 = False
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                level8_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    paragraph5 = False
                    paragraph6 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                    level8_no = 0
                else: 
                    paragraph6 = False
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    level8_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph_no = 0
            if penalty5:
                paragraph6 = False
                penalty5_roman = True
                level6_no += 1
                level6_string = str('number item ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                level8_no = 0
            elif penalty6:
                paragraph7 = False
                penalty6_roman = True
                level7_no += 1
                level7_string = str('number item ' + str(level7_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                level8_no = 0
            else: 
                if letters_before4:
                    paragraph6 = False
                    level6_no += 1
                    level6_string = str('number item ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                    level8_no = 0
                else:
                    paragraph7 = False
                    level7_no += 1
                    level7_string = str('number item ' + str(level7_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                    level8_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 4
            if level4_no==0 or bullets_before4:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 5
            elif level5_no==0 or bullets_before5:
                paragraph5 = False
                paragraph6 = False
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    paragraph6 = False
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            paragraph5 = False
                            paragraph6 = False
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                            level6_no = 0
                            level7_no = 0
                    # follwoing both roman numerals and lowercase
                    elif penalty5_roman:
                        if level6_no>1:
                            paragraph6 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_roman:
                        if level6_no>1:
                            paragraph6 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0
                    # following both roman numerals and lowercase
                    elif penalty6_roman:
                        if level7_no>1:
                            paragraph7 = False
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})
                            level8_no = 0





        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            paragraph3 = False
            paragraph4 = True
            paragraph5 = False
            paragraph6 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level5_no = 0
            level6_no = 0
            level7_no = 0
            paragraph_no = 0

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)):
                    penalty4 = False
                    penalty5 = False
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                    elif level4_no==0 or paragraph3:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        paragraph4 = False
                        paragraph5 = False
                        paragraph6 = False
                    else:
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level7_no = 0
                        paragraph5 = False
                        paragraph6 = False
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string]:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                        level4_no = 0
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        letters_before4 = False
                        bullets_before4 = False
                        bullets_before = False
                        penalty5 = False
                        penalty6 = False
                        paragraph3 = False
                        paragraph4 = False 
                        paragraph5 = False
                        paragraph6 = False
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        paragraph4 = False 
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        level6_no = 0
                        level7_no = 0
                        penalty5=0

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 0
            if penalty4:
                penalty4_roman = True
                roman_no = 2
                level5_no += 2
                level5_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['penalty sublaw i. '].append({'P ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
            elif penalty5:
                penalty5_roman = True
                roman_no = 2
                level6_no += 2
                level6_string = str('penalty sublaw ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-2]['penalty sublaw i. '].append({'P ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
            else: 
                if letters_before3:
                    roman_no = 2
                    level4_no += 2
                    level4_string = str('ii. ')
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-2]['i. '].append({'P':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                else:
                    roman_no = 2
                    level5_no += 2
                    level5_string = str('ii. ')
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-2]['i. '].append({'P':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 0
            if penalty4:
                penalty4_roman = True
                roman_no += 1
                level5_no += 1
                level5_string = str('penalty sublaw ' + str(roman_labels[roman_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
                paragraph5 = False
                paragraph6 = False
            elif penalty5:
                penalty5_roman = True
                roman_no += 1
                level6_no += 1
                level6_string = str('penalty sublaw ' + str(roman_labels[roman_no-1]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
            else: 
                if letters_before3:
                    roman_no += 1
                    level4_no += 1
                    level4_string = roman_labels[roman_no-1]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0 
                    level6_no = 0
                    level7_no = 0
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                else:
                    roman_no += 1
                    level5_no += 1
                    level5_string = roman_labels[roman_no-1]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 
                    level7_no = 0
                    paragraph5 = False
                    paragraph6 = False

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            penalty4_roman = False
            penalty5_roman = False
            paragraph_no = 0
            # no sub-points --> level 3
            if level3_no==0:
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # same level as lowercase --> level 4
            elif not in_alphabet:
                paragraph4 = False
                paragraph5 = False
                paragraph6 = False
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    paragraph4 = False
                    paragraph5 = False
                    paragraph6 = False
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    level6_no = 0
                    level7_no = 0
                else:
                    paragraph5 = False
                    paragraph6 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 0
            if penalty4:
                paragraph5 = False
                paragraph6 = False
                penalty4_roman = True
                level5_no += 1
                level5_string = str('number item ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                level7_no = 0
            elif penalty5:
                paragraph6 = False
                penalty5_roman = True
                level6_no += 1
                level6_string = str('number item ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                if letters_before3:
                    paragraph5 = False
                    paragraph6 = False
                    level5_no += 1
                    level5_string = str('number item ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 
                    level7_no = 0 
                else:
                    paragraph6 = False
                    level6_no += 1
                    level6_string = str('number item ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0  


        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives:
            in_bullet = True
            paragraph_no = 0
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                paragraph4 = False
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                level6_no = 0
                level7_no = 0
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    paragraph5 = False
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                    level7_no = 0
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_roman:
                        if level4_no>1:
                            paragraph4 = False
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                            level5_no = 0
                            level6_no = 0
                            level7_no = 0
                    # if only penalty letter
                    elif penalty4_roman:
                        if level5_no>1:
                            paragraph5 = False
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                            level6_no = 0
                            level7_no = 0
               # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            paragraph5 = False
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                            level6_no = 0
                            level7_no = 0
                    # if only penalty letter
                    elif penalty5_roman:
                        if level6_no>1:
                            paragraph6 = False
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                            level7_no = 0

        # adds paragraphs
        if (re.search(r'^(?!(Part|LAW|Law|\s|\((A|B|C|D|E)\)|Penalty:—$|\(i\))).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0 and level6_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5) and level6_no==0:
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and (level6_no==0 or paragraph6):
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0:
                paragraph7 = True
                paragraph_no += 1
                level7_no += 1
                level7_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string: []})





        # Ref notes
        if re.search('^((NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('REF NOTES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law|NOTE\.)\s', temp_text)) and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            #print(temp_text)

        if (re.search(r'^(?!((NOTES)|(LAW|Law)|(The Board)|\(|\s)).*', temp_text)) and level0_no>=1 and level2_no>=1 and in_directives:
            level3_no += 1
            level3_string = str('P' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
        
        if re.search(r'^\(([0-9][0-9]|[0-9])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1:
            level4_no += 1
            level4_string = str('point ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0

        if re.search(r'^\(([a-z])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1:
            if level4_no>=1:
                level5_no += 1
                level5_string = str('subpoint ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                level6_no = 0
            else:
                level4_no += 1
                level4_string = str('point ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
   

       
    yaml_data['(full ruleset)'][0]['(main ruleset)'].append({'Law 1 ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1954_1959 = (TEXTS_DIR / '1954-1959')
final_path_1954_1959 = (TREES_DIR / '1954-1959' / 'clean_')

for file_path in initial_path_1954_1959.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1954_1959(file, final_path_1954_1959, year)

In [ ]:
yaml_files_path_1954_1959 = (TREES_DIR / '1954-1959')

for file_path in yaml_files_path_1954_1959.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_1954_1959_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    letter_sections = False


    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

        # returns laws --> level 2
        if (re.search(r'^(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            letter_no = 0
            letter_sections = False
            bullets_before3 = False
            bullets_before4 = False
            bullets_before5 = False
            letters_before3 = False
            letters_before4 = False
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            penalty6 = False
            paragraph_no = 0

        # returns letter sections --> level 3 if present
        if re.search(fr'^\((A|B|C|D|E)\)\s', temp_text) and level2_no>=1 and not in_directives:
            letter_sections = True
            letter_no += 1
            level3_no += 1
            level3_string = str('LETTER ' + str(letter_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            penalty4 = False
            penalty5 = False
            paragraph_no = 1

        # returns defintions or sublaws -- letter sections present --> level 4
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            in_alphabet = False
            in_bullet = False
            penalty5 = False
            penalty6 = False
            letters_before4 = False
            bullets_before4 = False
            letters_before5 = False
            bullets_before5 = False
            ruling_possible = False
            ruling_real = False
            level4_no += 1
            level4_string = str('sublaw ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text) or re.search(r'^(P|F)\.K\.\s\([a-r]\)\s', temp_text) or re.search(r'^\([0-9]\)\s\([a-r]\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            bullets_before5 = False
            in_bullet = False
            paragraph_no = 1
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level4_no==0 or letters_before4:
                    letters_before4 = True
                    letters_before4 = True
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                else:
                    level5_no += 1
                    level5_string = str('subsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                    level6_no = 0
                    level6_no=0
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                paragraph_no = 1
                if level4_no==0 or letters_before4:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        letters_before4 = True
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string]: 
                        level5_no += 1
                        level5_string = str('subsublaw ' + str(level5_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
                        level6_no = 0
                        level6_no=0    

        # returns roman numerals (same level as lowercase --> level 5) -- letter sections present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty ii. ':[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                    level6_no = 0 
                else:
                    level6_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'ii. ':[]})
                    level7_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                level6_string = str('penalty ' + str(roman_labels[level6_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 1
                    level5_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = roman_labels[level6_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 6) -- letter sections present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            penalty5_roman = False
            penalty6_roman = False
            paragraph_no = 1
            # no sub-points --> level 4
            if level4_no==0:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # same level as lowercase --> level 5
            elif not in_alphabet:
                penalty5 = True
                level5_no += 1
                level5_string = str('penalty ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            # sublevel of lowercase --> level 6
            elif in_alphabet:
                if letters_before4:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else: 
                    penalty6 = True
                    level6_no += 1
                    level6_string = str('penalty ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and level4_no>=1 and letter_sections and not in_directives:
            paragraph_no = 1
            if penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('number item ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            elif penalty6:
                penalty6_roman = True
                level6_no += 1
                level6_string = str('number item ' + str(level6_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
            else: 
                if letters_before4:
                    level5_no += 1
                    level5_string = str('number item ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
                else:
                    level6_no += 1
                    level6_string = str('number item ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    level7_no = 0

        # returns bullets -- letter sections present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and level3_no>=1 and letter_sections and not in_directives:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 4
            if level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets BEFORE lowercase letters --> level 5
            elif level5_no==0 or bullets_before5:
                bullets_before5 = True
                level5_no += 1
                level5_string = str('subsubsublaw ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 6
                if not penalty5 and not penalty6:
                    level6_no += 1
                    level6_string = str('subsubsublaw ' + str(level6_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following numerical sublaws
                elif penalty5 and not penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # follwoing both roman numerals and lowercase
                    elif penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                # for penalties following lowercase sublaws
                elif penalty6:
                    # if no penalty roman numerals or letters
                    if not penalty6_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                    # following both roman numerals and lowercase
                    elif penalty6_roman:
                        if level7_no>1:
                            level7_no += 1
                            level7_string = str('penalty subsublaw ' + str(level7_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string][level6_no-1][level6_string].append({level7_string:[]})





        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text) or re.search(fr'^(F|P)\.K\.\s\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            in_alphabet = False
            in_bullet = False
            penalty4 = False
            penalty5 = False
            letters_before3 = False
            bullets_before3 = False
            letters_before4 = False
            bullets_before4 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 5
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            bullets_before4 = False
            in_bullet = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                # letters as sublaws of number sublaws
                if (not penalty4 or not re.search(r'^\(a\)\s', temp_text)) and (not penalty5 or not re.search(r'^\(a\)\s', temp_text)):
                    penalty4 = False
                    penalty5 = False 
                    in_alphabet = True
                    if level3_no==0 or letters_before3:
                        letters_before3 = True
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                    else:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0
            elif (re.match(r'\(i\)\s', temp_text)):
                # letters as sublaws of number sublaws
                in_alphabet = True
                if level3_no==0 or letters_before3:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string]:
                        letters_before3 = True
                        level3_no += 1
                        level3_string = str('subsublaw ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                else:
                    if {'subsublaw 8': []} in yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string]:
                        level4_no += 1
                        level4_string = str('subsublaw ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0
                        penalty5=0

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'ii. ':[]})
                    level5_no = 0
                else:
                    level5_no += 2
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'i. ':[]})
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'ii. ':[]})
                    level6_no = 0 
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 1
                    level4_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0 
                else:
                    level5_no += 1
                    level5_string = roman_labels[level5_no]
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 

        # returns penalties or exceptions (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text) or re.search(r'^(Exception:|Exceptions:)', temp_text)) and level2_no>=1 and not letter_sections and not in_directives:
            penalty4_roman = False
            penalty5_roman = False
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_alphabet:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_alphabet:
                if letters_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0
        
        # number sublist --> follows alphabet -- letter sections present
        if (re.search(r'^([1-9])\.\s', temp_text)) and level2_no>=1 and level3_no>=1 and not letter_sections and not in_directives:
            paragraph_no = 1
            if penalty4:
                penalty4_roman = True
                level4_no += 1
                level4_string = str('number item ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                penalty5_roman = True
                level5_no += 1
                level5_string = str('number item ' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
            else: 
                if letters_before3:
                    level4_no += 1
                    level4_string = str('number item ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('number item ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0   


        # returns bullets -- letter sections NOT present 
        if re.search(r'^(o|•|\*|-)\s', temp_text) and level2_no>=1 and not letter_sections and not in_directives:
            in_bullet = True
            paragraph_no = 1
            # bullets BEFORE numbers --> level 3
            if level3_no==0 or bullets_before3:
                bullets_before3 = True
                level3_no += 1
                level3_string = str('subsubsublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
            # bullets BEFORE lowercase letters --> level 4
            elif level4_no==0 or bullets_before4:
                bullets_before4 = True
                level4_no += 1
                level4_string = str('subsubsublaw ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
            # bullets AFTER lowercase letters or penalties 
            else: 
                # in law itself --> level 5
                if not penalty4 and not penalty5:
                    level5_no += 1
                    level5_string = str('subsubsublaw ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                # for penalties following numerical sublaws
                elif penalty4 and not penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty4_roman:
                        if level4_no>1:
                            level4_no += 1
                            level4_string = str('penalty subsublaw ' + str(level4_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    # if only penalty letter
                    elif penalty4_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
               # for penalties following lowercase sublaws
                elif penalty5:
                    # if no penalty roman numerals or letters
                    if not penalty5_roman:
                        if level5_no>1:
                            level5_no += 1
                            level5_string = str('penalty subsublaw ' + str(level5_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    # if only penalty letter
                    elif penalty5_roman:
                        if level6_no>1:
                            level6_no += 1
                            level6_string = str('penalty subsublaw ' + str(level6_no))
                            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalty|o|•|\*|-|\(|[0-9]|\s|NOTE|A\s|B\s|C\s|D\s|E\s)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if level3_no==0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1  
                if paragraph_no >=2:  
                    level2_no += 1
                    level2_string = str('LAW ' + str(real_law_no) + ' P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0 and level6_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no==0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0 and level6_no!=0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})



        # Ref notes
        if re.search('^((NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('REF NOTES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law|NOTE\.)', temp_text)) and in_directives:
            point_no += 1
            #print(temp_text)

        if (re.search(r'^(?!((NOTE)|(LAW|Law)|(The Board)|\(|\s)).*', temp_text)) and level0_no>=1 and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(point_no) + ' P' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            #print(temp_text)
        
        if re.search(r'^\(([0-9][0-9]|[0-9])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1:
            level3_no += 1
            level3_string = str('point ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0

        if re.search(r'^\(([a-z])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1 :
            if level3_no>=1:
                level4_no += 1
                level4_string = str('subpoint ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                level5_no = 0
            else:
                level3_no += 1
                level3_string = str('point ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0


            
                
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1954_1959 = (TEXTS_DIR / '1954-1959')
final_path_1954_1959flat = (TREES_DIR / '1954-1959flat' / 'clean_')

for file_path in initial_path_1954_1959.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1954_1959_flattened(file, final_path_1954_1959flat, year)

In [ ]:
yaml_files_path_1954_1959flat = (TREES_DIR / '1954-1959flat')

for file_path in yaml_files_path_1954_1959flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1934-1949

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1934-1949

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. LAW 1 + PART I + Notes (not in 1934) -- remove parts in flattened
 2. LAW #.s
 3. (a), (b) 
 4. (1), (2) -- sometimes skips (a)
 5. Note: or Penalty:- 
 6. (i), (ii) or (1), (2) 

 Notes section: NOTES FOR THE GUIDANCE OF REFEREES
  - LAW #
    - (a) or (1)
 
exception: law 34 has (a) -> (1) -- changed in text file to 1.
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
def create_yamls_1934_1949(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    paragraph3 = False
    paragraph4 = False
    paragraph5 = False
    paragraph6 = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

        # returns parts --> level 1
        if (re.search(r'^(Part)\s(I|II|III)', temp_text)) and not in_directives:
            level1_no += 1
            level1_string = str('part ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0

        # returns laws --> level 2
        if (re.search(r'^(?!LAW\s1\.)(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            numbers_before3 = False
            in_numbers = False
            penalty3 = False
            penalty4 = False
            penalty5 = False
            notes3 = False
            paragraph_no = 0
            paragraph3 = False
            pargraph4 = False
            paragraph5 = False
            paragraph6 = False

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives:
            in_numbers = False
            notes3 = False
            paragraph_no = 0
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                level5_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not in_directives:
            notes3 = False
            paragraph_no = 0
            if (not penalty3 or (penalty3 and not re.search(r'^\(1\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(1\)\s', temp_text)))and (not penalty5 or (penalty5 and not re.search(r'^\(1\)\s', temp_text))):
                penalty3 = False
                penalty4 = False
                penalty5 = False
                in_numbers = True
                if level3_no==0 or numbers_before3 or paragraph3:
                    numbers_before3 = True
                    numbers_before3 = True
                    level3_no += 1
                    level3_string = str('subsublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                    level4_no = 0
                    level5_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                else:
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0
                    penalty5=0
                    paragraph4 = False
                    paragraph5 = False
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                paragraph5 = False

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            paragraph_no = 0
            if penalty3:
                level4_no += 2
                level4_string = str('penalty sublevel ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty sublevel i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            elif penalty4:
                level5_no += 2
                level5_string = str('penalty sublevel ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty sublevel i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                paragraph5 = False
            elif penalty5:
                level6_no += 2
                level6_string = str('penalty sublevel ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({'penalty sublevel i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            paragraph_no = 0
            if penalty3:
                level4_no += 1
                level4_string = str('penalty sublevel ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            elif penalty4:
                level5_no += 1
                level5_string = str('penalty sublevel ' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                paragraph5 = False
            elif penalty5:
                level6_no += 1
                level6_string = str('penalty sublevel ' + str(roman_labels[level6_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string:[]})
                level7_no = 0
                paragraph6 = False


        # returns penalties  (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 0
            # no sub-points --> level 3
            if level3_no==0:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
            # same level as lowercase --> level 4
            elif not in_numbers:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            # sublevel of lowercase --> level 5
            elif in_numbers:
                if numbers_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
                else:
                    paragraph5 = False
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0  

        # returns notes 
        if (re.search(r'^(Note:)', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 0
            # no sub-points --> level 3
            if level3_no==0 or notes3:
                notes3 = True
                level3_no += 1
                level3_string = str('note ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                level5_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
            # same level as lowercase --> level 4
            elif not in_numbers:
                level4_no += 1
                level4_string = str('note ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            # sublevel of lowercase --> level 5
            elif in_numbers:
                paragraph5 = False
                if numbers_before3:
                    level4_no += 1
                    level4_string = str('note ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                    paragraph4 = False
                else:
                    level5_no += 1
                    level5_string = str('note ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 



        # adds paragraphs
        if (re.search(r'^(?!(Part|Law|LAW|NOTES|\s)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5):
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0:
                paragraph6 = True
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})




        # Ref notes
        if re.search('^((NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('REF NOTES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law)\s', temp_text)) and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            #print(temp_text)

        if (re.search(r'^(?!((NOTES)|(LAW|Law)|(The Board)|\(|\"|\s|Answer|Note)).*', temp_text)) and level0_no>=1 and level2_no>=1 and in_directives:
            level3_no += 1
            level3_string = str('P' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
        
        if re.search(r'^\(([0-9][0-9]|[0-9]|[a-z])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1:
            level4_no += 1
            level4_string = str('point ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0

        if re.search(r'^(Note)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1 and level4_no>=1:
            level5_no += 1
            level5_string = str('subpoint ' + str(level5_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            level6_no = 0
    
   

       
    yaml_data['(full ruleset)'][0]['(main ruleset)'].append({'Law 1 ': []})
   

           
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)

       



In [ ]:
initial_path_1934_1949 = (TEXTS_DIR / '1934-1949')
final_path_1934_1949 = (TREES_DIR / '1934-1949' / 'clean_')

for file_path in initial_path_1934_1949.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')

        create_yamls_1934_1949(file, final_path_1934_1949, year)

In [ ]:
yaml_files_path_1934_1949 = (TREES_DIR / '1934-1949')

for file_path in yaml_files_path_1934_1949.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:

def create_yamls_1934_1949_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # returns laws --> level 2
        if (re.search(r'^(?!LAW\s1\.)(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            numbers_before3 = False
            in_numbers = False
            penalty3 = False
            penalty4 = False
            penalty5 = False
            notes3 = False
            paragraph_no = 0

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives:
            in_numbers = False
            notes3 = False
            paragraph_no = 1
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                penalty4 = False
                penalty5 = False
                numbers_before3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0


        # returns defintions or sublaws -- letter sections NOT present --> level 3
        if (re.search(fr'^\(([0-9]|[0-9][0-9])\)\s(?!\()', temp_text)) and level2_no>=1 and not in_directives:
            notes3 = False
            paragraph_no = 1
            if (not penalty3 or (penalty3 and not re.search(r'^\(1\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(1\)\s', temp_text)))and (not penalty5 or (penalty5 and not re.search(r'^\(1\)\s', temp_text))):
                penalty3 = False
                penalty4 = False
                penalty5 = False
                in_numbers = True
                if level3_no==0 or numbers_before3:
                    numbers_before3 = True
                    numbers_before3 = True
                    level3_no += 1
                    level3_string = str('subsublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                else:
                    level4_no += 1
                    level4_string = str('subsublaw ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0
                    penalty5=0
            elif penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty' + str(level5_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            paragraph_no = 1
            if penalty3:
                level3_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({'penalty ii. ':[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty ii. ':[]})
                level6_no = 0
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            paragraph_no = 1
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(roman_labels[level3_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            elif penalty5:
                level5_no += 1
                level5_string = str('penalty' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0


        # returns penalties  (same level as lowercase --> level 5) -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_numbers:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_numbers:
                if numbers_before3:
                    penalty4 = True
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    penalty5 = True
                    level5_no += 1
                    level5_string = str('penalty ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0  

        # returns notes 
        if (re.search(r'^(Note:)', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0 or notes3:
                notes3 = True
                level3_no += 1
                level3_string = str('note ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            # same level as lowercase --> level 4
            elif not in_numbers:
                level4_no += 1
                level4_string = str('note ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
            # sublevel of lowercase --> level 5
            elif in_numbers:
                if numbers_before3:
                    level4_no += 1
                    level4_string = str('note ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                    level5_no = 0
                else:
                    level5_no += 1
                    level5_string = str('note ' + str(level5_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                    level6_no = 0 

        # adds paragraphs
        if (re.search(r'^(?!(Part|LAW|Penalty|i|v|\(|[0-9]|\s|NOTES|Note)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if level3_no==0 and level4_no==0 and level5_no==0:
                paragraph_no += 1
                if paragraph_no>=2:
                    level2_no += 1
                    level2_string = str('LAW ' + str(real_law_no) + ' P' + str(paragraph_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0 and level5_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no==0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})




        # Ref notes
        if re.search('^((NOTES FOR THE GUIDANCE OF REFEREES)|(NOTES ON THE LAWS OF THE GAME))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('REF NOTES ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level2_no = 0
            level3_no = 0
            level4_no = 0
            point_no = 0
            continue

        if (re.search(r'^(LAW|Law)\s', temp_text)) and in_directives:
            point_no += 1

        if (re.search(r'^(?!((NOTES)|(LAW|Law)|(The Board)|\(|\"|\s|Answer|Note)).*', temp_text)) and level0_no>=1 and in_directives:
            level2_no += 1
            level2_string = str('heading ' + str(point_no) + ' P' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
        
        if re.search(r'^\(([0-9][0-9]|[0-9]|[a-z])\)', temp_text) and in_directives and level0_no>=1 and level2_no>=1:
            level3_no += 1
            level3_string = str('point ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0

        if re.search(r'^(Note)', temp_text) and in_directives and level0_no>=1 and level2_no>=1 and level3_no>=1 :
            level4_no += 1
            level4_string = str('point ' + str(level4_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            level5_no = 0
    
   

       
    yaml_data['(full ruleset)'][0]['(main ruleset)'].append({'Law 1 ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1934_1949 = (TEXTS_DIR / '1934-1949')
final_path_1934_1949flat = (TREES_DIR / '1934-1949flat' / 'clean_')

for file_path in initial_path_1934_1949.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        #print(year)
        create_yamls_1934_1949_flattened(file, final_path_1934_1949flat, year)

In [ ]:
yaml_files_path_1934_1949flat = (TREES_DIR / '1934-1949flat')

for file_path in yaml_files_path_1934_1949flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1929

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1929

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. LAW 1 + PART I + IRB -- remove parts in flattened
 2. LAW #.s
 3. (a), (b) 
 4. (i), (ii)
 5. Penalty: -- sometimes before (a)
 6. (a), (b) in penalty 

 Notes section:CIRCULAR LETTER OF THE INTERNATIONAL RUGBY FOOTBALL BOARD
  - LAW # or Month
    - (1)
      - (a)
 
exception: law 34 has (a) -> (1) -- changed in text file to 1.
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
def create_yamls_1929(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    level5_no = 0
    level6_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False
    paragraph3 = False
    paragraph4 = False

    level0_no = 1
    level0_string = str('(main rulebook)')
    yaml_data['(full ruleset)'].append({level0_string: []})
            

    for line in file:
        temp_text = line

        # returns parts --> level 1
        if (re.search(r'^(Part)\s(I|II|III)', temp_text)) and not in_directives:
            level1_no += 1
            level1_string = str('part ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False

        # returns laws --> level 2
        if (re.search(r'^(?!LAW\s1\.)(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            penalty3 = False
            penalty4 = False
            notes3 = False
            paragraph_no = 0
            paragraph3 = False
            paragraph4 = False
            paragraph5 = False

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 0
            paragraph3 = False
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                if (not penalty3 or (penalty3 and not re.search(r'^\(a\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(a\)\s', temp_text))):
                    penalty3 = False
                    penalty4 = False
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                elif penalty3:
                    level3_no += 1
                    level3_string = str('penalty ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                elif penalty4:
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False
            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                if not penalty3 and not penalty4:
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                elif penalty3:
                    level3_no += 1
                    level3_string = str('penalty ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
                    paragraph3 = False
                    paragraph4 = False
                    paragraph5 = False
                elif penalty4:
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0
                    paragraph4 = False
                    paragraph5 = False

        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            paragrah_no = 0
            paragraph3 = False
            if penalty3:
                level4_no += 1
                level4_string = 'penalty sublaw ii. '
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            elif penalty4:
                level5_no += 1
                level5_string = 'penalty sublaw ii. '
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({'penalty sublaw i. ':[]})
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                paragraph5 = False
        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            if penalty3:
                level4_no += 1
                level4_string = str('penalty sublaw ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False
            elif penalty4:
                level5_no += 1
                level5_string = str('penalty sublaw ' + str(roman_labels[level5_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string:[]})
                level6_no = 0
                paragraph5 = False


        # returns penalties -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 0
            # no sub-points --> level 3
            if level3_no==0 or penalty3 or paragraph3:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
                paragraph3 = False
                paragraph4 = False
                paragraph5 = False
            # sublevel of lowercase --> level 4
            else:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
                paragraph4 = False
                paragraph5 = False

        # adds paragraphs
        if (re.search(r'^(?!(LAW|Part|\s|CIRCULAR|Penalty)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if (level3_no==0 or paragraph3) and level4_no==0 and level5_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4) and level5_no==0:
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0 and (level5_no==0 or paragraph5):
                paragraph5 = True
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})
            elif level3_no!=0 and level4_no!=0 and level5_no!=0:
                paragraph_no += 1
                level6_no += 1
                level6_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string][level5_no-1][level5_string].append({level6_string: []})



        # IRFB letter
        if re.search('^(CIRCULAR LETTER OF THE INTERNATIONAL RUGBY FOOTBALL BOARD)', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('IRB Letter ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level1_no = 0
            continue

        if (re.search(r'^((LAW)|(The Board))', temp_text)) and level0_no>=1 and in_directives:
            level1_no += 1
            level1_string = str('heading' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0

        if (re.search(r'^(?!((CIRCULAR LETTER)|(LAW)|(The Board)|\([a-z]|\s)).*', temp_text)) and level0_no>=1 and level1_no>=1 and in_directives:
            level2_no += 1
            level2_string = str('P' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
        
        if (re.search(r'^(\([a-z])', temp_text)) and level0_no>=1 and level1_no>=1 and level2_no>=1 and in_directives:
            level3_no += 1
            level3_string = str('point' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0
            level4_string = str('P1')
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
        
    

       
    yaml_data['(full ruleset)'][0]['(main rulebook)'].append({'Law 1 ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)


In [ ]:
initial_path_1929 = (TEXTS_DIR / '1929')
final_path_1929 = (TREES_DIR / '1929' / 'clean_')

for file_path in initial_path_1929.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1929(file, final_path_1929, year)

In [ ]:
yaml_files_path_1929 = (TREES_DIR / '1929')

for file_path in yaml_files_path_1929.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_1929_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    penalty4 = False
    penalty5 = False
    penalty6 = False
    in_directives = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line


        # returns laws --> level 2
        if (re.search(r'^(?!LAW\s1\.)(LAW)\s[0-9]', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            level5_no = 0
            level6_no = 0
            penalty3 = False
            penalty4 = False

            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^\([a-r]\)\s', temp_text)) and level2_no>=1 and not in_directives:
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            if not re.match(r'\(i\)\s', temp_text):
                if (not penalty3 or (penalty3 and not re.search(r'^\(a\)\s', temp_text))) and (not penalty4 or (penalty4 and not re.search(r'^\(a\)\s', temp_text))):
                    penalty3 = False
                    penalty4 = False
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0

                    paragraph_no = 1

                elif penalty3 and (penalty_letters or re.search(r'^\(a\)\s', temp_text)):
                    penalty_letters = True
                    if not re.search(r'^\(a\)\s', temp_text):
                        level3_no += 1
                        level3_string = str('penalty ' + str(level3_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                        level4_no = 0

                elif penalty4 and (penalty_letters or re.search(r'^\(a\)\s', temp_text)):
                    penalty_letters = True
                    if not re.search(r'^\(a\)\s', temp_text):
                        level4_no += 1
                        level4_string = str('penalty ' + str(level4_no))
                        yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                        level5_no = 0

            elif (re.match(r'\(i\)\s', temp_text)) and not penalty3 and not penalty4 and not penalty5:
                if not penalty3 and not penalty4:
                    numbers_before3 = False
                    level3_no += 1
                    level3_string = str('sublaw ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0

                    paragraph_no = 1

                elif penalty3 and (penalty_letters):
                    level3_no += 1
                    level3_string = str('penalty ' + str(level3_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0

                elif penalty4 and penalty_letters:
                    level4_no += 1
                    level4_string = str('penalty ' + str(level4_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
                    level5_no = 0


        
        # returns roman numerals (same level as lowercase --> level 4) -- letter sections NOT present 
        roman_labels = ['i. ', 'ii. ', 'iii. ', 'iv. ', 'v. ', 'vi. ', 'vii. ', 'viii. ', 'ix. ', 'x. ', 'xi. ', 'xii. ', 'xiii. ', 'xiv. ', 'xv. ', 'xvi. ', 'xvii. ', 'xviii. ', 'xix. ', 'xx. ', 'xxi. ']
        # only adds i. if there is a ii., so doesn't add lowercase letter i. 
        if (re.search(r'^\(ii\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0

                paragraph_no = 1

            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ii. ')
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({'penalty ii. ':[]})
                level5_no = 0

                paragraph_no = 1

        if (re.search(r'\((iii|iv|v|vi|vii|viii|ix|x|xi|xii|xiii|xiv|xv|xvi|xvii|xviii|xxix|xx|xxi|xxii)\)\s', temp_text)) and level2_no>=1 and level3_no>=1 and not in_directives:
            if penalty3:
                level3_no += 1
                level3_string = str('penalty ' + str(roman_labels[level3_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0

                paragraph_no = 1

            elif penalty4:
                level4_no += 1
                level4_string = str('penalty ' + str(roman_labels[level4_no]))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0

                paragraph_no = 1



        # returns penalties -- letter sections NOT present 
        if (re.search(r'^(Penalty:)', temp_text)) and level2_no>=1 and not in_directives:
            penalty_letters = False
            # no sub-points --> level 3
            if level3_no==0 or penalty3:
                penalty3 = True
                level3_no += 1
                level3_string = str('penalty ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0

            # sublevel of lowercase --> level 4
            else:
                penalty4 = True
                level4_no += 1
                level4_string = str('penalty ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0


        # adds paragraphs
        if (re.search(r'^(?!(LAW|Penalty|i|v|\(|[0-9]|\s|CIRCULAR)).*', temp_text)) and level2_no>=1 and real_law_no>=1 and not in_directives:
            if level3_no==0 and level4_no==0:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})


            
        # IRFB letter
        if re.search('^(CIRCULAR LETTER OF THE INTERNATIONAL RUGBY FOOTBALL BOARD)', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('IRB Letter ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level1_no = 0
            continue

        if (re.search(r'^(?!((CIRCULAR LETTER)|(LAW)|(The Board)|\(|\"|\s|Answer)).*', temp_text)) and level0_no>=1 and in_directives:
            level1_no += 1
            level1_string = str('P' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0
        
        if (re.search(r'^(\([0-9]|\")', temp_text)) and level0_no>=1 and level1_no>=1 and in_directives:
            level2_no += 1
            level2_string = str('point' + str(level2_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
        
        if (re.search(r'^(\([a-z])', temp_text)) and level0_no>=1 and level1_no>=1 and level2_no>=1 and in_directives:
            level3_no += 1
            level3_string = str('subpoint' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0

    

       
    yaml_data['(full ruleset)'][0]['(main ruleset)'].append({'Law 1 ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1929 = (TEXTS_DIR / '1929')
final_path_1929flat = (TREES_DIR / '1929flat' / 'clean_')

for file_path in initial_path_1929.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
        
        create_yamls_1929_flattened(file, final_path_1929flat, year)

In [ ]:
yaml_files_path_1929flat = (TREES_DIR / '1929flat')

for file_path in yaml_files_path_1929flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1894-1924

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1894-1924

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. field + I + IRB -- remove parts in flattened
 2. 1.-, 2.- 
 3. (a), (b)
 4. Note.- -- sometimes before (a) 

 Notes section: INTERNATIONAL RUGBY FOOTBALL BOARD (not in 1894)
 
exception: law 34 has (a) -> (1) -- changed in text file to 1.
'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
# adding everywhere
def create_yamls_1894_1924(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    notes3 = False
    paragraph3 = False
    paragraph4 = False
    in_directives = False

    level0_no  = 1
    level0_string = str('(main ruleset)') 
    yaml_data['(full ruleset)'].append({level0_string: []})

    for line in file:
        temp_text = line

        # returns parts --> level 1
        if (re.search(r'^(I|II|III|IV|V)\.\s', temp_text)) and not in_directives:
            level1_no += 1
            level1_string = str('part ' + str(level1_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
            level2_no = 0
            paragraph2 = False
            paragraph3 = False
            paragraph4 = False

        # returns laws --> level 2
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and level1_no>=1 and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            notes3 = False
            paragraph_no = 0
            paragraph2 = False
            paragraph3 = False
            paragraph4 = False

        # returns lowercase letter -- letter sections NOT present --> level 3 
        if (re.search(r'^(\([a-r]\)|\([a-r]\.\))\s', temp_text)) and level2_no>=1 and not in_directives:
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
            notes3 = False
            paragraph3 = False
            paragraph4 = False
            level3_no += 1
            level3_string = str('sublaw ' + str(level3_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            level4_no = 0


        # returns notes -- letter sections NOT present 
        if (re.search(r'^((Note)|(NOTE))\.', temp_text)) and level2_no>=1 and not in_directives:
            # no sub-points --> level 3
            if level3_no==0 or notes3 or paragraph3:
                paragraph3 = False
                paragraph4 = False
                notes3 = True
                level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0

            # sublevel of lowercase --> level 4
            else:
                paragraph4 = False
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0


        # add paragraphs
        if (re.search(r'^(?!(I|V|\s|INTER|At|--)).*', temp_text)) and level1_no>=1 and level2_no>=1 and real_law_no>=1 and not in_directives:
            if (level3_no==0 or paragraph3) and level4_no==0:
                paragraph3 = True
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and (level4_no==0 or paragraph4):
                paragraph4 = True
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})
            elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level5_no += 1
                level5_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string][level4_no-1][level4_string].append({level5_string: []})

        
        # Ref notes
        if re.search('^((INTERNATIONAL RUGBY FOOTBALL BOARD)|(INTERNATIONAL BOARD)|(At the Annual General Meeting)|(Hon\.))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('IRB Letter ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level1_no = 0
            continue

        if (re.search(r'^(?!(INTERNATIONAL|\s|At the Annual|--)).*', temp_text)) and level0_no>=1 and in_directives:
            if year!='1894' and year!='1904':
                if len(temp_text)<=40:
                    level1_no += 1
                    level1_string = str('heading' + str(level1_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
                    level2_no = 0
            elif year=='1894' or year=='1904':
                level1_no += 1
                level1_string = str('P' + str(level1_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
                level2_no = 0

        if (re.search(r'^(?!(INTERNATIONAL|\s|At the Annual|\(|\*|--)).*', temp_text)) and level0_no>=1 and level1_no>=1 and in_directives and len(temp_text)>40:
            if year!='1894' and year!='1904':
                level2_no += 1
                level2_string = str('P' + str(level2_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0
        
        if (re.search(r'^(\(|\*|--)', temp_text)) and level0_no>=1 and level1_no>=1 and level2_no>=1 and in_directives:
            if year!='1894' and year!='1904':
                level3_no += 1
                level3_string = str('point' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            else:
                level2_no += 1
                level2_string = str('P' + str(level2_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0
        
  
    #yaml_data['(full ruleset)'][0]['(main ruleset)'] = fill_empty_levels(yaml_data['(full ruleset)'][0]['(main ruleset)'])

    yaml_data['(full ruleset)'].append({'field ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1894_1924 = (TEXTS_DIR / '1894-1924')
final_path_1894_1924 = (TREES_DIR / '1894-1924' / 'clean_')

for file_path in initial_path_1894_1924.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')

        create_yamls_1894_1924(file, final_path_1894_1924, year)

In [ ]:
yaml_files_path_1894_1924 = (TREES_DIR / '1894-1924')

for file_path in yaml_files_path_1894_1924.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_1894_1924_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level1_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    notes3 = False
    in_directives = False

    level0_no = 1
    level0_string = str('(main ruleset)')
    yaml_data['(full ruleset)'].append({level0_string: []})


    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            level3_no = 0
            level4_no = 0
            notes3 = False
            paragraph_no = 1

        # returns lowercase letter -- letter sections NOT present --> level 3 + 4
        if (re.search(r'^(\([a-r]\)|\([a-r]\.\))\s', temp_text)) and level2_no>=1 and not in_directives:
            # anytime it reads i., it adds as next letter (even in case of 6.29.a.i processing as 6.29.b)
            # remove falsely registered i. as different alphabetic character
                notes3 = False
                level3_no += 1
                level3_string = str('sublaw ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
                paragraph_no = 1


        # returns notes -- letter sections NOT present 
        if (re.search(r'^((Note)|(NOTE))\.', temp_text)) and level2_no>=1 and not in_directives:
            paragraph_no = 1
            # no sub-points --> level 3
            if level3_no==0 or notes3:
                notes3 = True
                level3_no += 1
                level3_string = str('notes ' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string:[]})
                level4_no = 0
            # sublevel of lowercase --> level 4
            else:
                level4_no += 1
                level4_string = str('notes ' + str(level4_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string:[]})
                level5_no = 0
        
        if (re.search(r'^(?!(Note|NOTE|\#|\*|I|V|\(|[0-9]|\s|INTER|At)).*', temp_text)) and real_law_no>=1 and not in_directives:
            if level3_no==0 and level4_no==0:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('LAW ' + str(real_law_no) + 'P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level2_string: []})
            elif level3_no!=0 and level4_no==0:
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string].append({level3_string: []})
            elif level3_no!=0 and level4_no!=0:
                paragraph_no += 1
                level4_no += 1
                level4_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})


 
        # IRFB letter
        if re.search('^((INTERNATIONAL RUGBY FOOTBALL BOARD)|(INTERNATIONAL BOARD)|(At the Annual General Meeting)|(Hon\.))', temp_text):
            in_directives = True
            level0_no = 2
            level0_string = str('IRB Letter ')
            yaml_data['(full ruleset)'].append({level0_string: []})
            level1_no = 0
            continue

        if (re.search(r'^(?!(INTERNATIONAL|\s|At the Annual|--)).*', temp_text)) and level0_no>=1 and in_directives:
            if year!='1894' and year!='1904':
                if len(temp_text)<=40:
                    level1_no += 1
                    level1_string = str('heading' + str(level1_no))
                    yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
                    level2_no = 0
            elif year=='1894' or year=='1904':
                level1_no += 1
                level1_string = str('P' + str(level1_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string].append({level1_string: []})
                level2_no = 0

        if (re.search(r'^(?!(INTERNATIONAL|\s|At the Annual|\(|\*|--)).*', temp_text)) and level0_no>=1 and level1_no>=1 and in_directives and len(temp_text)>40:
            if year!='1894' and year!='1904':
                level2_no += 1
                level2_string = str('P' + str(level2_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0
        
        if (re.search(r'^(\(|\*|--)', temp_text)) and level0_no>=1 and level1_no>=1 and level2_no>=1 and in_directives:
            if year!='1894' and year!='1904':
                level3_no += 1
                level3_string = str('point' + str(level3_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            else:
                level2_no += 1
                level2_string = str('P' + str(level2_no))
                yaml_data['(full ruleset)'][level0_no-1][level0_string][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0

       
    yaml_data['(full ruleset)'].append({'field ': []})
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1894_1924 = (TEXTS_DIR / '1894-1924')
final_path_1894_1924flat = (TREES_DIR / '1894-1924flat' / 'clean_')

for file_path in initial_path_1894_1924.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1894_1924_flattened(file, final_path_1894_1924flat, year)

In [ ]:
yaml_files_path_1894_1924flat = (TREES_DIR / '1894-1924flat')

for file_path in yaml_files_path_1894_1924flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1874-1889

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1874-1889

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. field + 1.- (ref regulation + cases -- only in 1889, but not in text file)
 2. (1), (2), (i.), (ii.) in paragraphs 


'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
def create_yamls_1874_1889(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    paragraph_no = 1
    in_directives = False

    if year=='1889':
        level1_no  = 1
        level1_string = str('(main ruleset)') 
        yaml_data['(full ruleset)'].append({level1_string: []})   

        for line in file:
            temp_text = line

            # returns laws --> level 2
            if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
                real_law_no += 1
                level2_no += 1
                level2_string = str('LAW ' + str(real_law_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string].append({level2_string: []})
                paragraph_no = 0
                level3_no = 0
            
            if (re.search(r'^(?!(\s|CASE)).*', temp_text)) and real_law_no>=1 and not in_directives:
                #print(temp_text)
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            
            # returns subsections --> level 4
            if (re.search(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)) and level2_no>=1 and not in_directives:
                sublaws = re.findall(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)
                for sublaw in sublaws:
                    level3_no += 1
                    level3_string = str('sublaw ' + sublaw)
                    yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 1
                    level4_string = str('P ' + str(level4_no))
                    yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})

            
            # Ref notes
            if re.search('^(CASE LAW)', temp_text):
                in_directives = True
                level1_no  += 1
                level1_string = str('(case law)') 
                yaml_data['(full ruleset)'].append({level1_string: []}) 
                paragraph_no = 0
                level2_no = 0

            if (re.search(r'^(?!(POINTS|Decided|CASE|-|\(|\s)).*', temp_text)) and level1_no>=1 and in_directives:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0

            if (re.search(r'^(\(|-)', temp_text)) and level2_no>=1 and in_directives:
                level3_no += 1
                level3_string = str('subpoint' + str(level3_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 1
                level4_string = str('P ' + str(level4_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})


    else:
        yaml_data['(full ruleset)'].append({'(main ruleset)': []})

        for line in file:
            temp_text = line

            # returns laws --> level 2
            if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
                real_law_no += 1
                level2_no += 1
                level2_string = str('LAW ' + str(real_law_no))
                yaml_data['(full ruleset)'][0]['(main ruleset)'].append({level2_string: []})
                paragraph_no = 0
                level3_no = 0
            
            if (re.search(r'^(?!(\s|CASE)).*', temp_text)) and real_law_no>=1 and not in_directives:
                #print(temp_text)
                paragraph_no += 1
                level3_no += 1
                level3_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][0]['(main ruleset)'][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
            
            # returns subsections --> level 4
            if (re.search(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)) and level2_no>=1 and not in_directives:
                sublaws = re.findall(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)
                for sublaw in sublaws:
                    level3_no += 1
                    level3_string = str('sublaw ' + sublaw)
                    yaml_data['(full ruleset)'][0]['(main ruleset)'][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 1
                    level4_string = str('P ' + str(level4_no))
                    yaml_data['(full ruleset)'][0]['(main ruleset)'][level2_no-1][level2_string][level3_no-1][level3_string].append({level4_string: []})


       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1874_1889 = (TEXTS_DIR / '1874-1889')
final_path_1874_1889 = (TREES_DIR / '1874-1889' / 'clean_')

for file_path in initial_path_1874_1889.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1874_1889(file, final_path_1874_1889, year)

In [ ]:
yaml_files_path_1874_1889 = (TREES_DIR / '1874-1889')

for file_path in yaml_files_path_1874_1889.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_1874_1889_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0
    level3_no = 0
    level4_no = 0
    paragraph_no = 1
    in_directives = False

    if year=='1889': 
        level1_no  = 1
        level1_string = str('(main ruleset)') 
        yaml_data['(full ruleset)'].append({level1_string: []}) 

        for line in file:
            temp_text = line

            # identifies laws
            if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
                real_law_no += 1
                paragraph_no = 0
        
            # gets paragraphs
            if (re.search(r'^(?!(\s|CASE)).*', temp_text)) and real_law_no>=1 and not in_directives:
                #print(temp_text)
                paragraph_no += 1
                level2_no += 1
                level2_string = str('LAW ' + str(real_law_no) + ' P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0

            # returns subsections --> level 3
            if (re.search(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)) and level2_no>=1 and not in_directives:
                sublaws = re.findall(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)
                for sublaw in sublaws:
                    level3_no += 1
                    level3_string = str('sublaw ' + sublaw)
                    yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
            
            # case law
            if re.search('^(CASE LAW)', temp_text):
                in_directives = True
                level1_no  += 1
                level1_string = str('(case law)') 
                yaml_data['(full ruleset)'].append({level1_string: []}) 
                paragraph_no = 0
                level2_no = 0

            if (re.search(r'^(?!(POINTS|Decided|CASE|\s)).*', temp_text)) and level1_no>=1 and in_directives:
                paragraph_no += 1
                level2_no += 1
                level2_string = str('P' + str(paragraph_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string].append({level2_string: []})
                level3_no = 0

            if (re.search(r'^(\(|-)', temp_text)) and level2_no>=1 and in_directives:
                level3_no += 1
                level3_string = str('subpoint' + str(paragraph_no))
                yaml_data['(full ruleset)'][level1_no-1][level1_string][level2_no-1][level2_string].append({level3_string: []})
                level4_no = 0
        

    else:
        yaml_data['(full ruleset)'].append({'(main ruleset)': []})

        for line in file:
            temp_text = line

            # identifies laws
            if (re.search(r'^([0-9]|[0-9][0-9])\.', temp_text)) and not in_directives:
                real_law_no += 1
                paragraph_no = 0
        
            # gets paragraphs
            if (re.search(r'^(?!(\s|CASE)).*', temp_text)) and real_law_no>=1 and not in_directives:
                #print(temp_text)
                paragraph_no += 1
                level2_no += 1
                level2_string = str('LAW ' + str(real_law_no) + ' P' + str(paragraph_no))
                yaml_data['(full ruleset)'][0]['(main ruleset)'].append({level2_string: []})
                level3_no = 0

            # returns subsections --> level 3
            if (re.search(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)) and level2_no>=1 and not in_directives:
                sublaws = re.findall(r'\(([0-9]|i\.|ii\.|iii\.|iv\.|v\.)\)', temp_text)
                for sublaw in sublaws:
                    level3_no += 1
                    level3_string = str('sublaw ' + sublaw)
                    yaml_data['(full ruleset)'][0]['(main ruleset)'][level2_no-1][level2_string].append({level3_string: []})
                    level4_no = 0
            

       
    
    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1874_1889 = (TEXTS_DIR / '1874-1889')
final_path_1874_1889flat = (TREES_DIR / '1874-1889flat' / 'clean_')

for file_path in initial_path_1874_1889.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1874_1889_flattened(file, final_path_1874_1889flat, year)

In [ ]:
yaml_files_path_1874_1889flat = (TREES_DIR / '1874-1889flat')

for file_path in yaml_files_path_1874_1889flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## 1845

The author's notes on this era's heading/depth pattern:

In [ ]:

# 1845

# identify pattern of differentiators + filter
# note paragraphs

'''
calculate depths: 
 1. i., ii. 


'''
# add to dataset at appropriate depth


**Hierarchical (nested) version** — parse, save, then reload and visualize.

In [ ]:
def create_yamls_1845(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 1
    level2_no = 0

    yaml_data['(full ruleset)'].append({'(main ruleset)': []})

    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'(i|v|x)\.', temp_text)):
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][0]['(main ruleset)'].append({level2_string: []})
            yaml_data['(full ruleset)'][0]['(main ruleset)'][level2_no-1][level2_string].append({'P1': []})

    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1845 = (TEXTS_DIR / '1845')
final_path_1845 = (TREES_DIR / '1845' / 'clean_')

for file_path in initial_path_1845.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1845(file, final_path_1845, year)

In [ ]:
yaml_files_path_1845 = (TREES_DIR / '1845')

for file_path in yaml_files_path_1845.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

**Flattened version** of the same range — fewer collapsed heading levels.

In [ ]:
def create_yamls_1845_flattened(file, save_path, year):
    yaml_data = {'(full ruleset)': []}
    real_law_no = 0
    level2_no = 0

    yaml_data['(full ruleset)'].append({'(main ruleset)': []})

    for line in file:
        temp_text = line

         # returns laws --> level 2
        if (re.search(r'(i|v|x)\.', temp_text)):
            real_law_no += 1
            level2_no += 1
            level2_string = str('LAW ' + str(real_law_no))
            yaml_data['(full ruleset)'][0]['(main ruleset)'].append({level2_string: []})

    final_file = Path(str(save_path) + f'{year}tree.yaml')
    with open(final_file, 'w') as file:
        yaml.dump(yaml_data, file, default_flow_style=False, sort_keys=False)



In [ ]:
initial_path_1845 = (TEXTS_DIR / '1845')
final_path_1845flat = (TREES_DIR / '1845flat' / 'clean_')

for file_path in initial_path_1845.glob('*.txt'):
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('clean_', '')
       
        create_yamls_1845_flattened(file, final_path_1845flat, year)

In [ ]:
yaml_files_path_1845flat = (TREES_DIR / '1845flat')

for file_path in yaml_files_path_1845flat.glob('*.yaml'):
    print(file_path)
    with open(file_path, 'r') as file:
        file_name = file_path.stem
        year = file_name.replace('tree', '')
        year = year.replace('clean_', '')

        data = yaml.safe_load(file)
        G = create_heirarchical_tree(data)

        generate_graph(G, year)

## Comparison grids

`generate_graph_comparison` draws every year's tree from a folder of saved
YAML files into one big grid figure (11 rows x 4 columns), for comparing
structure across years at a glance.

In [ ]:
def generate_graph_comparison(file_path):

    fig, axes = plt.subplots(nrows = 11, ncols = 4, figsize=(28,77))
    flat_axes = axes.flatten()
    
    for i, file_path in enumerate(sorted(file_path.glob('*.yaml'))):
        with open(file_path, 'r') as file:
            file_name = file_path.stem
            year = file_name.replace('tree', '')
            year = year.replace('clean_', '')

            data = yaml.safe_load(file)
            G = create_heirarchical_tree(data)

            pos = graphviz_layout(G, prog='twopi', root='center', args='-Groot="center" -Gnodesep=0.4 -Granksep=1')
            node_colors = [G.nodes[node]['color'] for node in G.nodes()]
            node_sizes = [G.nodes[node]['node_size'] for node in G.nodes()]

            ax = flat_axes[i]

            nx.draw(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, width=1, edge_color='dimgrey')

            # make size of axes so doesn't pull diagram
            xs = [p[0] for p in pos.values()]
            ys = [p[1] for p in pos.values()]
            max_range = max(max(xs) - min(xs), max(ys) - min(ys))
            center_x = (min(xs) + max(xs)) / 2
            center_y = (min(ys) + max(ys)) / 2
            padding = 0.05 * max_range
            half = (max_range / 2) + padding
            ax.set_xlim(center_x - half, center_x + half)
            ax.set_ylim(center_y - half, center_y + half)
            ax.set_aspect("equal")

            ax.set_title(str(year), size=30)

    plt.tight_layout()
    return fig, axes


Grid of every flattened tree.

In [ ]:
yaml_files_path_allflat = (TREES_BASE / 'tree_structures_paragraphs' / 'all_flat')

generate_graph_comparison(yaml_files_path_allflat)

Grid of every un-flattened (hierarchical) tree.

In [ ]:
yaml_files_path_allunflat = (TREES_BASE / 'tree_structures_paragraphs' / 'all_unflat')

generate_graph_comparison(yaml_files_path_allunflat)